# LeetCode 265 — Paint House II (Hard) bằng Python

> **Lưu ý: đây là bài Premium.** Trên LeetCode, bài 265 bị khoá — cần tài khoản **LeetCode Premium** mới xem được đề và nộp bài. Đề bài, ví dụ, constraints và signature trong notebook này được trình bày theo bản đề hiện tại của LeetCode (theo hiểu biết của người soạn). Bạn **không cần Premium** để học hết notebook: mọi lời giải đều được kiểm thử ngay tại đây, kể cả đối chiếu với brute force.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minCostII(self, costs: List[List[int]]) -> int:

```

Bài này là **bản nâng cấp** của LeetCode 256 — Paint House: bài 256 chỉ có đúng **3 màu**, còn bài 265 có **`k` màu**. Với 3 màu, ta có thể viết tay 3 công thức; với `k` màu thì phải tổng quát hoá. Đề còn có câu **follow-up** (câu hỏi mở rộng): *"Bạn có làm được trong thời gian `O(nk)` không?"* — và đó chính là phần "Hard" thật sự của bài.

Lộ trình của notebook đi đúng con đường một người mới sẽ đi: thử **tham lam** (và thấy nó sai) → **thử hết** mọi cách tô (đúng nhưng quá chậm) → **đệ quy + ghi nhớ** → **bảng DP** → phát hiện ra chỉ cần **2 giá trị nhỏ nhất** → cuối cùng chỉ còn **3 biến**.

---

## Sau notebook này, bạn sẽ nắm được

1. Đọc và thao tác **ma trận chi phí** `costs` (list of lists), `min`, `enumerate`, `float("inf")`.
2. Vì sao cách **tham lam (greedy)** "mỗi nhà chọn màu rẻ nhất có thể" lại **sai** — có phản ví dụ chạy được.
3. **Cách 1 — Brute force**: dùng `itertools.product` sinh đủ `k^n` cách tô, lọc ràng buộc "hai nhà kề không cùng màu".
4. **Đệ quy** và **memoization** (ghi nhớ) từ con số 0; **Cách 2 — đệ quy + memo** theo trạng thái `(i, màu nhà trước)`.
5. **Ôn LeetCode 256 (k = 3)** từ căn bản, rồi tổng quát thành **Cách 3 — DP bảng** `dp[i][c]`, thời gian `O(nk²)`.
6. **Bước đột phá**: "giá trị nhỏ nhất của một hàng khi bỏ đi một vị trí" chỉ cần biết **2 giá trị nhỏ nhất** → **Cách 4 — `O(nk)`** (cách chính), có trace đầy đủ và chứng minh.
7. **Cách 5** — chỉ giữ **3 biến** `min1, min2, idx1`: bộ nhớ thêm `O(1)`.
8. `class Solution` để nộp, kiểm thử chéo với brute force, vì sao đúng, lỗi hay gặp, mẹo nhận dạng pattern, các biến thể (LeetCode 1289, dãy nhà vòng tròn...) và **6 bài tập** có gợi ý + đáp án.

## 1. Đề bài (theo LeetCode)

Có một dãy `n` ngôi nhà, mỗi nhà có thể được sơn bằng **một trong `k` màu**. Chi phí sơn mỗi nhà bằng mỗi màu là khác nhau. Bạn phải sơn **tất cả** các nhà sao cho **không có hai nhà kề nhau nào cùng màu**.

Chi phí sơn được cho bởi ma trận `costs` kích thước `n x k`.

- Ví dụ, `costs[0][0]` là chi phí sơn nhà `0` bằng màu `0`;
- `costs[1][2]` là chi phí sơn nhà `1` bằng màu `2`, và cứ thế...

Hãy trả về **chi phí nhỏ nhất** để sơn tất cả các nhà.

**Example 1**

```text
Input:  costs = [[1,5,3],[2,9,4]]
Output: 5
Explanation:
Paint house 0 into color 0, paint house 1 into color 2. Minimum cost: 1 + 4 = 5;
Or paint house 0 into color 2, paint house 1 into color 0. Minimum cost: 3 + 2 = 5.
```

**Example 2**

```text
Input:  costs = [[1,3],[2,4]]
Output: 5
```

**Constraints**

- `costs.length == n`
- `costs[i].length == k`
- `1 <= n <= 100`
- `2 <= k <= 20`
- `1 <= costs[i][j] <= 20`

**Follow up:** Could you solve it in `O(nk)` runtime?

Những điều cần để ý ngay từ đầu:

- **"Kề nhau"** chỉ là nhà `i` và nhà `i + 1`. Nhà `0` và nhà `2` **được phép** cùng màu.
- Vì `k >= 2`, **luôn có** ít nhất một cách tô hợp lệ (ví dụ tô xen kẽ màu 0, 1, 0, 1, ...). Nên đáp án luôn là một số hữu hạn.
- Nếu `n = 1` thì không có cặp kề nào: chỉ cần chọn màu rẻ nhất của nhà duy nhất.
- Mọi chi phí đều **dương** (từ 1 đến 20), đáp án lớn nhất cũng chỉ `100 * 20 = 2000`.
- Ta đặt tên: **`n` = số nhà** (số hàng của `costs`), **`k` = số màu** (số cột). Hai chữ này dùng suốt notebook khi nói về độ phức tạp.

In [1]:
from typing import List   # LeetCode tự import sẵn; chạy ngoài LeetCode thì ta tự import
import itertools
import random

costs_ex1 = [[1, 5, 3],
             [2, 9, 4]]
costs_ex2 = [[1, 3],
             [2, 4]]

# Ví dụ tự tạo, dùng XUYÊN SUỐT notebook để trace: 4 nhà x 4 màu, có chỗ "hoà" (2 giá trị bằng nhau)
costs_demo = [[3, 1, 4, 2],
              [2, 7, 1, 8],
              [5, 2, 6, 2],
              [4, 1, 3, 5]]

print("Example 1:", costs_ex1, "-> mong đợi 5")
print("Example 2:", costs_ex2, "-> mong đợi 5")
print("Example 1: n (số nhà) =", len(costs_ex1), "| k (số màu) =", len(costs_ex1[0]))
print("costs_demo: n =", len(costs_demo), "| k =", len(costs_demo[0]))

Example 1: [[1, 5, 3], [2, 9, 4]] -> mong đợi 5
Example 2: [[1, 3], [2, 4]] -> mong đợi 5
Example 1: n (số nhà) = 2 | k (số màu) = 3
costs_demo: n = 4 | k = 4


### 1.1. Làm tay Example 1 trước khi viết bất kỳ dòng code nào

Viết ma trận thành bảng: **mỗi hàng là một nhà, mỗi cột là một màu**.

| | màu 0 | màu 1 | màu 2 |
|---|---:|---:|---:|
| **nhà 0** | 1 | 5 | 3 |
| **nhà 1** | 2 | 9 | 4 |

Một **cách tô** là một danh sách màu, ví dụ `(0, 2)` nghĩa là nhà 0 màu 0, nhà 1 màu 2. Với 2 nhà và 3 màu có `3 x 3 = 9` cách tô:

| màu nhà 0 | màu nhà 1 | hợp lệ? | chi phí |
|---:|---:|---|---:|
| 0 | 0 | không (trùng màu) | — |
| 0 | 1 | có | 1 + 9 = 10 |
| 0 | 2 | có | 1 + 4 = **5** |
| 1 | 0 | có | 5 + 2 = 7 |
| 1 | 1 | không (trùng màu) | — |
| 1 | 2 | có | 5 + 4 = 9 |
| 2 | 0 | có | 3 + 2 = **5** |
| 2 | 1 | có | 3 + 9 = 12 |
| 2 | 2 | không (trùng màu) | — |

Có 6 cách hợp lệ, nhỏ nhất là **5** (đạt được bằng 2 cách — khớp với phần Explanation của đề). Để ý: nếu **không** có ràng buộc, ta chỉ việc chọn màu rẻ nhất mỗi nhà: `1 + 2 = 3`. Nhưng cả hai nhà đều rẻ nhất ở **màu 0** → trùng màu → bị cấm. Toàn bộ độ khó của bài nằm ở ràng buộc này.

In [2]:
# Liệt kê 9 cách tô của Example 1 bằng 2 vòng for lồng nhau
k = len(costs_ex1[0])
for c0 in range(k):            # màu của nhà 0
    for c1 in range(k):        # màu của nhà 1
        if c0 == c1:
            print(f"({c0}, {c1}) -> không hợp lệ (trùng màu)")
        else:
            total = costs_ex1[0][c0] + costs_ex1[1][c1]
            print(f"({c0}, {c1}) -> {costs_ex1[0][c0]} + {costs_ex1[1][c1]} = {total}")

(0, 0) -> không hợp lệ (trùng màu)
(0, 1) -> 1 + 9 = 10
(0, 2) -> 1 + 4 = 5
(1, 0) -> 5 + 2 = 7
(1, 1) -> không hợp lệ (trùng màu)
(1, 2) -> 5 + 4 = 9
(2, 0) -> 3 + 2 = 5
(2, 1) -> 3 + 9 = 12
(2, 2) -> không hợp lệ (trùng màu)


## 2. Kiến thức nền 1: ma trận chi phí, `min`, `enumerate`, `float("inf")`

**Ma trận** trong Python là một **list chứa các list con** (list of lists). Ở bài này:

- `costs[i]` là **hàng thứ `i`** = bảng giá của **nhà `i`** cho mọi màu.
- `costs[i][c]` là giá sơn **nhà `i`** bằng **màu `c`** (đếm từ 0).
- `n = len(costs)` (số nhà), `k = len(costs[0])` (số màu).

Vài công cụ sẽ dùng liên tục:

| Cú pháp | Đọc là | Ví dụ với `row = [1, 5, 3]` |
|---|---|---|
| `min(row)` | giá trị nhỏ nhất | `1` |
| `row.index(x)` | vị trí **đầu tiên** của `x` | `row.index(1) = 0` |
| `min(range(k), key=lambda c: row[c])` | **vị trí** của giá trị nhỏ nhất (argmin) | `0` |
| `enumerate(row)` | duyệt lấy cả **chỉ số** lẫn **giá trị** | `(0, 1), (1, 5), (2, 3)` |
| `float("inf")` | "vô cùng" — lớn hơn mọi số | `min(inf, 7) = 7` |
| `min(row[c] for c in range(k) if c != 0)` | min của các màu **khác màu 0** | `min(5, 3) = 3` |

Hai dòng cuối rất quan trọng:

- `float("inf")` (ta sẽ đặt tên `INF`) dùng làm **giá trị khởi đầu** khi đi tìm min: số nào gặp đầu tiên cũng nhỏ hơn `INF`. Ngoài ra `INF + 5` vẫn là `INF`, rất tiện để biểu diễn "không thể".
- `min(... for c in range(k) if c != 0)` là một **generator expression** (biểu thức sinh): nó duyệt `c = 0, 1, ..., k-1`, **bỏ qua** các `c` không thoả điều kiện `if`, rồi đưa phần còn lại cho `min`. Đây chính là phép "min của các màu khác màu `c`" — trái tim của bài toán.
- `lambda c: row[c]` là một **hàm vô danh** viết gọn: nhận `c`, trả về `row[c]`. Truyền nó vào `key=` để `min` so sánh theo `row[c]` nhưng **trả về `c`**.

In [3]:
row = costs_ex1[0]                     # bảng giá của nhà 0
k = len(row)
print("costs_ex1[0]    =", row)
print("costs_ex1[1][2] =", costs_ex1[1][2], "(nhà 1, màu 2)")
print("min(row)        =", min(row))
print("row.index(1)    =", row.index(1))
print("argmin          =", min(range(k), key=lambda c: row[c]))

for c, cost in enumerate(row):
    print(f"  màu {c}: giá {cost}")

INF = float("inf")
print("INF > 10**18 ?", INF > 10**18, "| min(INF, 7) =", min(INF, 7), "| INF + 5 =", INF + 5)
print("min các màu khác màu 0:", min(row[c] for c in range(k) if c != 0))
print("min các màu khác màu 2:", min(row[c] for c in range(k) if c != 2))

costs_ex1[0]    = [1, 5, 3]
costs_ex1[1][2] = 4 (nhà 1, màu 2)
min(row)        = 1
row.index(1)    = 0
argmin          = 0
  màu 0: giá 1
  màu 1: giá 5
  màu 2: giá 3
INF > 10**18 ? True | min(INF, 7) = 7 | INF + 5 = inf
min các màu khác màu 0: 3
min các màu khác màu 2: 1


## 3. Kiến thức nền 2: `itertools.product`, `zip` và `all` — sinh và kiểm tra mọi cách tô

Ở mục 1.1 ta dùng 2 vòng `for` lồng nhau cho 2 nhà. Nhưng nếu có `n` nhà thì cần `n` vòng lồng nhau — không viết tay được khi `n` thay đổi. Thư viện chuẩn có sẵn công cụ:

```python
itertools.product(range(k), repeat=n)
```

sinh ra **mọi bộ `n` phần tử**, mỗi phần tử lấy từ `range(k)` — đúng bằng mọi cách tô `n` nhà bằng `k` màu. Tổng cộng có `k^n` bộ (mỗi nhà `k` lựa chọn, nhân lên).

Để kiểm tra một cách tô `colors` có hợp lệ không, ta cần so **từng cặp kề nhau** `(colors[0], colors[1])`, `(colors[1], colors[2])`, ...

- `colors[1:]` là **slicing** (cắt list): bỏ phần tử đầu. Với `colors = (0, 2, 2)` thì `colors[1:] = (2, 2)`.
- `zip(colors, colors[1:])` ghép từng phần tử với phần tử **đứng ngay sau nó**: `(0, 2), (2, 2)`.
- `all(...)` trả `True` khi **mọi** điều kiện bên trong đều đúng.

Vậy `all(a != b for a, b in zip(colors, colors[1:]))` đọc là: "mọi cặp kề nhau đều khác màu".

In [4]:
combos = list(itertools.product(range(3), repeat=2))     # 2 nhà, 3 màu
print("Số cách tô 2 nhà x 3 màu:", len(combos), "=", 3 ** 2)
print(combos)

colors = (0, 2, 2)
print("colors        =", colors)
print("colors[1:]    =", colors[1:])
print("các cặp kề    =", list(zip(colors, colors[1:])))
print("hợp lệ?       =", all(a != b for a, b in zip(colors, colors[1:])))
print("(0, 2, 0) hợp lệ?", all(a != b for a, b in zip((0, 2, 0), (2, 0))))   # nhà 0 và 2 cùng màu: vẫn OK

Số cách tô 2 nhà x 3 màu: 9 = 9
[(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]
colors        = (0, 2, 2)
colors[1:]    = (2, 2)
các cặp kề    = [(0, 2), (2, 2)]
hợp lệ?       = False
(0, 2, 0) hợp lệ? True


## 4. Ý tưởng đầu tiên của người mới: tham lam (greedy)

**Tham lam (greedy)** nghĩa là: ở mỗi bước, chọn cái **trông có vẻ tốt nhất ngay lúc đó**, không bao giờ quay lại sửa.

Áp vào bài này, cách nghĩ tự nhiên nhất là:

1. Nhà 0: chọn màu **rẻ nhất**.
2. Nhà 1: chọn màu rẻ nhất **khác màu nhà 0**.
3. Nhà 2: chọn màu rẻ nhất **khác màu nhà 1**.
4. ... cứ thế đến hết.

Cách này rất nhanh (`O(nk)`) và — bất ngờ thay — cho **đúng cả 2 ví dụ của đề**. Nhưng "đúng ví dụ" chưa phải là "đúng". Hãy chạy thử với một bộ dữ liệu được thiết kế để "gài bẫy":

```text
costs_trap = [[1, 2, 30],
              [1, 30, 30]]
```

In [5]:
def min_cost_greedy(costs):
    """THAM LAM (SAI): mỗi nhà chọn màu rẻ nhất khác màu nhà trước."""
    total, prev_color, chosen = 0, -1, []
    for row in costs:
        best_c = min((c for c in range(len(row)) if c != prev_color), key=lambda c: row[c])
        total += row[best_c]
        chosen.append(best_c)
        prev_color = best_c
    return total, chosen


costs_trap = [[1, 2, 30],
              [1, 30, 30]]

print("Example 1 :", min_cost_greedy(costs_ex1), "-> đúng là 5")
print("Example 2 :", min_cost_greedy(costs_ex2), "-> đúng là 5")
print("costs_trap:", min_cost_greedy(costs_trap), "-> đúng là 3 (nhà 0 màu 1, nhà 1 màu 0)")
print("costs_demo:", min_cost_greedy(costs_demo), "-> đúng là 5 (sẽ kiểm bằng brute force)")

Example 1 : (5, [0, 2]) -> đúng là 5
Example 2 : (5, [0, 1]) -> đúng là 5
costs_trap: (31, [0, 1]) -> đúng là 3 (nhà 0 màu 1, nhà 1 màu 0)
costs_demo: (7, [1, 2, 1, 2]) -> đúng là 5 (sẽ kiểm bằng brute force)


### Vì sao tham lam sai?

Nhìn `costs_trap`:

- Tham lam thấy nhà 0 rẻ nhất ở **màu 0** (giá 1) → chọn ngay.
- Nhưng nhà 1 cũng rẻ nhất ở **màu 0** (giá 1) — giờ bị cấm! Hai màu còn lại đều giá 30 → tổng `1 + 30 = 31`.
- Nếu nhà 0 "hy sinh" một chút, chọn **màu 1** (giá 2), thì nhà 1 được dùng màu 0 (giá 1) → tổng `2 + 1 = 3`. Tốt hơn **gấp 10 lần**.

**Bài học:** lựa chọn ở nhà `i` **ảnh hưởng tới các nhà sau** (nó khoá mất một màu của nhà `i + 1`). Một lựa chọn "rẻ trước mắt" có thể khiến tương lai đắt đỏ. Vì vậy ta **không được** chốt màu sớm — phải **cân nhắc nhiều khả năng cùng lúc**. Đây là dấu hiệu kinh điển của bài toán **quy hoạch động (Dynamic Programming — DP)**.

Nhưng trước khi tới DP, ta cần một lời giải **chắc chắn đúng** để làm "trọng tài" kiểm tra mọi cách sau: **brute force**.

## 5. Cách 1 — Brute force: thử mọi cách tô

Ý tưởng đơn giản nhất mà **chắc chắn đúng**:

1. Sinh **mọi** cách tô bằng `itertools.product(range(k), repeat=n)`.
2. Bỏ những cách có hai nhà kề cùng màu.
3. Với mỗi cách còn lại, cộng chi phí: `sum(costs[i][colors[i]] for i in range(n))`.
4. Lấy giá trị nhỏ nhất.

Vì xét **hết** mọi khả năng, không thể bỏ sót đáp án — nên brute force là "trọng tài" hoàn hảo cho các cách nhanh hơn (chỉ dùng được với input nhỏ).

In [6]:
def min_cost_bruteforce(costs):
    """Cách 1: thử đủ k^n cách tô, giữ lại cách hợp lệ rẻ nhất."""
    n, k = len(costs), len(costs[0])
    best = float("inf")
    for colors in itertools.product(range(k), repeat=n):
        if all(a != b for a, b in zip(colors, colors[1:])):        # không có 2 nhà kề cùng màu
            total = sum(costs[i][colors[i]] for i in range(n))
            best = min(best, total)
    return best


print("Example 1 :", min_cost_bruteforce(costs_ex1))
print("Example 2 :", min_cost_bruteforce(costs_ex2))
print("costs_trap:", min_cost_bruteforce(costs_trap))
print("costs_demo:", min_cost_bruteforce(costs_demo))
assert min_cost_bruteforce(costs_ex1) == 5
assert min_cost_bruteforce(costs_ex2) == 5
assert min_cost_bruteforce(costs_trap) == 3
assert min_cost_bruteforce(costs_demo) == 5

Example 1 : 5
Example 2 : 5
costs_trap: 3
costs_demo: 5


In [7]:
def trace_bruteforce(costs):
    """In từng cách tô, đánh dấu hợp lệ / không và cập nhật 'best' theo thời gian."""
    n, k = len(costs), len(costs[0])
    best = float("inf")
    for colors in itertools.product(range(k), repeat=n):
        if not all(a != b for a, b in zip(colors, colors[1:])):
            print(f"{colors}: bỏ (có 2 nhà kề cùng màu)")
            continue
        total = sum(costs[i][colors[i]] for i in range(n))
        note = "  <- best mới!" if total < best else ""
        best = min(best, total)
        print(f"{colors}: tổng = {total:>2} | best = {best}{note}")
    return best


trace_bruteforce(costs_ex1)

(0, 0): bỏ (có 2 nhà kề cùng màu)
(0, 1): tổng = 10 | best = 10  <- best mới!
(0, 2): tổng =  5 | best = 5  <- best mới!
(1, 0): tổng =  7 | best = 5
(1, 1): bỏ (có 2 nhà kề cùng màu)
(1, 2): tổng =  9 | best = 5
(2, 0): tổng =  5 | best = 5
(2, 1): tổng = 12 | best = 5
(2, 2): bỏ (có 2 nhà kề cùng màu)


5

### Cách 1 vướng ở đâu?

Số cách tô là `k^n` — **tăng theo hàm mũ** của số nhà. Mỗi cách lại tốn `O(n)` để kiểm tra và cộng tổng, nên thời gian là `O(n · k^n)`.

- `n = 2, k = 3`: 9 cách — làm tay được.
- `costs_demo` (`n = 4, k = 4`): 256 cách — máy vẫn nhanh.
- Constraints tối đa `n = 100, k = 20`: `20^100` cách — một con số có **131 chữ số**. Kể cả máy tính nhanh nhất thế giới chạy từ thuở Big Bang tới nay cũng không xong.

Điểm lãng phí: rất nhiều cách tô có **chung phần đầu**. Ví dụ `(1, 2, 3, 0)` và `(1, 2, 3, 1)` đều bắt đầu bằng `(1, 2, 3)`, nhưng brute force tính lại tổng của phần đầu đó từ đầu cho mỗi cách. Ta cần một cách **tái sử dụng** kết quả của phần đầu (hoặc phần sau) — đó là việc của đệ quy + ghi nhớ.

In [8]:
print(f"{'n':>4} {'k':>4} | số cách tô k^n")
for n, k in [(2, 3), (4, 4), (10, 5), (20, 10), (100, 20)]:
    count = k ** n
    shown = f"{count:,}" if count < 10**15 else f"khoảng 10^{len(str(count)) - 1} ({len(str(count))} chữ số)"
    print(f"{n:>4} {k:>4} | {shown}")

   n    k | số cách tô k^n
   2    3 | 9
   4    4 | 256
  10    5 | 9,765,625
  20   10 | khoảng 10^20 (21 chữ số)
 100   20 | khoảng 10^130 (131 chữ số)


## 6. Kiến thức nền 3: đệ quy (recursion) từ con số 0

**Đệ quy** là khi một hàm **gọi lại chính nó** với một bài toán **nhỏ hơn**. Mọi hàm đệ quy đều có hai phần:

1. **Trường hợp cơ sở (base case)**: bài toán nhỏ tới mức trả lời ngay được, **không gọi tiếp**. Thiếu phần này hàm sẽ gọi mãi mãi.
2. **Bước đệ quy**: tách bài toán thành "một việc nhỏ làm ngay" + "phần còn lại, giao cho chính hàm này".

Ví dụ mini: tính tổng các phần tử từ vị trí `i` đến hết list `nums`.

- Base case: `i == len(nums)` → không còn phần tử nào → tổng là `0`.
- Bước đệ quy: `suffix_sum(i) = nums[i] + suffix_sum(i + 1)`.

Với `nums = [4, 1, 3]`: `suffix_sum(0) = 4 + suffix_sum(1) = 4 + (1 + suffix_sum(2)) = 4 + (1 + (3 + suffix_sum(3))) = 4 + 1 + 3 + 0 = 8`.

Máy tính nhớ các lời gọi đang dở dang trong một **ngăn xếp lời gọi (call stack)**: lời gọi `suffix_sum(0)` phải **chờ** `suffix_sum(1)` trả lời, `suffix_sum(1)` lại chờ `suffix_sum(2)`, ... Cell dưới in thụt lề theo độ sâu để bạn **nhìn thấy** call stack đi xuống rồi đi lên.

Để ý dạng của hàm: **"đáp án từ vị trí `i` đến hết"**. Lát nữa lời giải đệ quy của Paint House cũng có đúng dạng đó.

In [9]:
def suffix_sum(nums, i, depth=0):
    pad = "    " * depth
    if i == len(nums):                      # base case
        print(f"{pad}suffix_sum({i}) = 0   (hết phần tử)")
        return 0
    print(f"{pad}suffix_sum({i}) = {nums[i]} + suffix_sum({i + 1}) ...")
    result = nums[i] + suffix_sum(nums, i + 1, depth + 1)   # bước đệ quy
    print(f"{pad}suffix_sum({i}) = {result}")
    return result


suffix_sum([4, 1, 3], 0)

suffix_sum(0) = 4 + suffix_sum(1) ...
    suffix_sum(1) = 1 + suffix_sum(2) ...
        suffix_sum(2) = 3 + suffix_sum(3) ...
            suffix_sum(3) = 0   (hết phần tử)
        suffix_sum(2) = 3
    suffix_sum(1) = 4
suffix_sum(0) = 8


8

## 7. Cách 2 — Đệ quy theo trạng thái `(i, màu nhà trước)` + memo

### Định nghĩa bài toán con

Thay vì chọn cả dãy màu một lúc, ta quyết định **từng nhà một**, từ trái sang phải. Khi đứng ở nhà `i`, điều duy nhất ta cần biết về quá khứ là **nhà `i - 1` đã tô màu gì** (để tránh trùng). Mọi chi tiết khác của quá khứ đều không ảnh hưởng tới tương lai!

Định nghĩa:

> `best(i, prev)` = chi phí nhỏ nhất để tô **các nhà `i, i+1, ..., n-1`**, biết rằng nhà `i - 1` đã được tô màu `prev` (quy ước `prev = -1` khi chưa có nhà nào trước, tức là lúc `i = 0`).

- **Base case**: `best(n, prev) = 0` — đã tô hết nhà, không tốn thêm gì.
- **Bước đệ quy**: thử mọi màu `c` khác `prev` cho nhà `i`, trả `costs[i][c]`, rồi giao phần còn lại cho `best(i + 1, c)`:

```text
best(i, prev) = min( costs[i][c] + best(i + 1, c)   với mọi c != prev )
```

- **Đáp án** = `best(0, -1)`.

### Ví dụ số với `costs_demo`

Nhà cuối (`i = 3`) có giá `[4, 1, 3, 5]`, và `best(4, ·) = 0`, nên `best(3, prev)` = màu rẻ nhất **khác `prev`**:

| `prev` | các màu được chọn | `best(3, prev)` |
|---:|---|---:|
| 0 | màu 1, 2, 3 → giá 1, 3, 5 | **1** |
| 1 | màu 0, 2, 3 → giá 4, 3, 5 | **3** |
| 2 | màu 0, 1, 3 → giá 4, 1, 5 | **1** |
| 3 | màu 0, 1, 2 → giá 4, 1, 3 | **1** |

Chỉ khi nhà 2 dùng màu 1 (chính màu rẻ nhất của nhà 3) thì nhà 3 mới phải trả đắt hơn (3 thay vì 1). Phần còn lại máy sẽ tính. Trước hết, viết bản đệ quy **chưa có ghi nhớ** và đếm số lần gọi hàm.

In [10]:
def min_cost_recursive(costs):
    """Cách 2a: đệ quy thuần (CHƯA ghi nhớ). Trả về (đáp án, số lần gọi hàm)."""
    n, k = len(costs), len(costs[0])
    calls = 0

    def best(i, prev):
        nonlocal calls            # cho phép hàm con sửa biến calls của hàm ngoài
        calls += 1
        if i == n:                # base case: tô hết rồi
            return 0
        return min(costs[i][c] + best(i + 1, c) for c in range(k) if c != prev)

    return best(0, -1), calls


print("Example 1 :", min_cost_recursive(costs_ex1))
print("costs_demo:", min_cost_recursive(costs_demo))

Example 1 : (5, 10)
costs_demo: (5, 161)


### Vấn đề: bài toán con bị tính đi tính lại

Với `costs_demo` (chỉ 4 nhà!), hàm đã bị gọi **161 lần**. Vì sao? Hãy nhìn `best(2, 0)` ("tô nhà 2, 3 khi nhà 1 màu 0"). Nó được gọi từ `best(1, 1)`, `best(1, 2)` và `best(1, 3)` — tức là **3 lần**, và mỗi lần lại tính lại toàn bộ cây con bên dưới. Hiện tượng này gọi là **bài toán con trùng lặp (overlapping subproblems)**, và nó làm số lời gọi tăng theo hàm mũ: khoảng `k · (k-1)^(n-1)`.

Nhưng chú ý: `best(i, prev)` chỉ phụ thuộc vào **hai số** `i` và `prev`. Có tất cả `n` giá trị `i` và `k + 1` giá trị `prev` (tính cả `-1`), tức là **chỉ có khoảng `n · (k + 1)` bài toán con khác nhau**. Nếu ta **ghi nhớ** đáp án của mỗi bài toán con ở lần tính đầu tiên, các lần sau chỉ việc tra lại.

### Kiến thức nền 4: memoization (ghi nhớ) bằng `dict`

**Memoization** = "học thuộc bài cũ": lưu kết quả của mỗi lời gọi vào một `dict` (key là tham số, value là kết quả). Trước khi tính, kiểm tra xem đã có trong `dict` chưa.

Ví dụ mini kinh điển: dãy Fibonacci `fib(n) = fib(n-1) + fib(n-2)`, với `fib(0) = 0, fib(1) = 1`. Bản thuần tính `fib(18)` hai lần, `fib(17)` ba lần, ... Python còn có sẵn decorator `functools.lru_cache` (hoặc `functools.cache`) làm đúng việc này tự động — nhưng tự viết bằng `dict` một lần sẽ giúp bạn hiểu nó làm gì bên trong.

In [11]:
from functools import lru_cache

calls_naive = 0
def fib_naive(n):
    global calls_naive
    calls_naive += 1
    return n if n < 2 else fib_naive(n - 1) + fib_naive(n - 2)


memo = {}
calls_memo = 0
def fib_memo(n):
    global calls_memo
    calls_memo += 1
    if n in memo:                 # đã tính rồi -> tra lại, không tính nữa
        return memo[n]
    result = n if n < 2 else fib_memo(n - 1) + fib_memo(n - 2)
    memo[n] = result              # ghi nhớ trước khi trả về
    return result


@lru_cache(maxsize=None)          # Python tự lo phần ghi nhớ
def fib_cache(n):
    return n if n < 2 else fib_cache(n - 1) + fib_cache(n - 2)


print("fib_naive(20) =", fib_naive(20), "| số lần gọi:", calls_naive)
print("fib_memo(20)  =", fib_memo(20), "| số lần gọi:", calls_memo)
print("fib_cache(20) =", fib_cache(20), "| thông tin cache:", fib_cache.cache_info())

fib_naive(20) = 6765 | số lần gọi: 21891
fib_memo(20)  = 6765 | số lần gọi: 39
fib_cache(20) = 6765 | thông tin cache: CacheInfo(hits=18, misses=21, maxsize=None, currsize=21)


### Áp dụng memo vào `best(i, prev)`

Chỉ cần thêm 3 dòng: tạo `memo = {}`, tra `memo` trước khi tính, lưu vào `memo` sau khi tính. Key là cặp `(i, prev)` — Python cho phép dùng **tuple** làm key của `dict`.

Độ phức tạp:

- Số trạng thái khác nhau: khoảng `n · (k + 1)`.
- Mỗi trạng thái chạy một vòng `for c in range(k)`: `O(k)` việc.
- Tổng thời gian: `O(n · k · k) = O(nk²)`. Bộ nhớ: `O(nk)` cho `memo` + `O(n)` cho độ sâu đệ quy (call stack).

Với `n <= 100` thì độ sâu đệ quy tối đa khoảng 101 — dưới giới hạn mặc định của Python (1000), nên an toàn.

In [12]:
def min_cost_memo(costs):
    """Cách 2b: đệ quy + memo theo trạng thái (i, prev). Trả về (đáp án, số lần gọi, memo)."""
    n, k = len(costs), len(costs[0])
    memo = {}
    calls = 0

    def best(i, prev):
        nonlocal calls
        calls += 1
        if i == n:
            return 0
        if (i, prev) in memo:                  # bài toán con đã giải -> tra lại
            return memo[(i, prev)]
        result = min(costs[i][c] + best(i + 1, c) for c in range(k) if c != prev)
        memo[(i, prev)] = result               # ghi nhớ
        return result

    return best(0, -1), calls, memo


ans, calls, memo_demo = min_cost_memo(costs_demo)
print("costs_demo: đáp án =", ans, "| số lần gọi có memo =", calls, "| số trạng thái đã lưu =", len(memo_demo))

print("\nSố lần gọi khi tăng số nhà (k = 4, chi phí ngẫu nhiên):")
random.seed(1)
for n in [4, 6, 8, 10]:
    cs = [[random.randint(1, 20) for _ in range(4)] for _ in range(n)]
    a1, c1 = min_cost_recursive(cs)
    a2, c2, _ = min_cost_memo(cs)
    assert a1 == a2
    print(f"  n = {n:>2}: không memo {c1:>7,} lần | có memo {c2:>4} lần")

costs_demo: đáp án = 5 | số lần gọi có memo = 41 | số trạng thái đã lưu = 13

Số lần gọi khi tăng số nhà (k = 4, chi phí ngẫu nhiên):
  n =  4: không memo     161 lần | có memo   41 lần
  n =  6: không memo   1,457 lần | có memo   65 lần
  n =  8: không memo  13,121 lần | có memo   89 lần
  n = 10: không memo 118,097 lần | có memo  113 lần


In [13]:
def show_memo(memo, n, k):
    """In memo dưới dạng bảng: hàng = i, cột = prev ('.' = trạng thái không cần tới)."""
    print("       " + "".join(f"prev={p:>2} " for p in range(-1, k)))
    for i in range(n):
        cells = [f"{memo[(i, p)]:>7} " if (i, p) in memo else "      . " for p in range(-1, k)]
        print(f"i = {i}: " + "".join(cells))


show_memo(memo_demo, len(costs_demo), len(costs_demo[0]))

       prev=-1 prev= 0 prev= 1 prev= 2 prev= 3 
i = 0:       5       .       .       .       . 
i = 1:       .       4       4       5       4 
i = 2:       .       3       3       3       5 
i = 3:       .       1       3       1       1 


### Đọc bảng memo

- Hàng `i = 3` đúng như bảng tính tay: `best(3, 0) = 1`, `best(3, 1) = 3`, `best(3, 2) = 1`, `best(3, 3) = 1`.
- Hàng `i = 0` chỉ có cột `prev = -1`: đó chính là **đáp án** `best(0, -1) = 5`.
- Ở hàng `i = 0`, các cột `prev = 0..3` là dấu `.` vì không ai hỏi tới (nhà 0 không có nhà trước).

### Cách 2 vướng ở đâu?

Cách 2 đã từ hàm mũ xuống `O(nk²)` — với constraints của đề (`100 · 20 · 20 = 40,000` việc) là **đủ để AC** (Accepted). Nhưng:

1. Đệ quy có chi phí phụ (mỗi lời gọi hàm, tạo tuple key, tra `dict`) và phụ thuộc độ sâu call stack.
2. Vẫn là `O(nk²)`, chưa đạt follow-up `O(nk)`.

Bước tiếp theo: lật ngược cách nghĩ, tính **từ dưới lên (bottom-up)** bằng một **bảng DP** — không cần đệ quy. Để làm quen, ta ôn lại bài anh em với 3 màu: **LeetCode 256**.

## 8. Ôn LeetCode 256 — Paint House (k = 3) từ căn bản

Bài 256 giống hệt bài 265, chỉ khác là **luôn có đúng 3 màu**: đỏ (0), xanh dương (1), xanh lá (2). Signature: `def minCost(self, costs: List[List[int]]) -> int`, constraints `1 <= n <= 100`, `costs[i].length == 3`, `1 <= costs[i][j] <= 20`.

Example 1 của bài 256: `costs = [[17,2,17],[16,16,5],[14,3,19]]` → `10` (nhà 0 xanh dương 2, nhà 1 xanh lá 5, nhà 2 xanh dương 3: `2 + 5 + 3 = 10`). Example 2: `[[7,6,2]]` → `2`.

### Định nghĩa DP theo hướng "từ trái sang"

Lần này ta định nghĩa theo **tiền tố** (các nhà từ 0 tới `i`) thay vì hậu tố như Cách 2:

> `dp[i][c]` = chi phí nhỏ nhất để tô **các nhà `0, 1, ..., i`** sao cho **nhà `i` có màu `c`**.

- Nhà đầu: `dp[0][c] = costs[0][c]` (chỉ có một nhà, tô màu `c` thì trả đúng giá đó).
- Nhà `i` màu đỏ (0) thì nhà `i - 1` phải là xanh dương hoặc xanh lá, và ta chọn phương án rẻ hơn:

```text
dp[i][0] = costs[i][0] + min(dp[i-1][1], dp[i-1][2])
dp[i][1] = costs[i][1] + min(dp[i-1][0], dp[i-1][2])
dp[i][2] = costs[i][2] + min(dp[i-1][0], dp[i-1][1])
```

- Đáp án = `min(dp[n-1][0], dp[n-1][1], dp[n-1][2])` — nhà cuối màu gì cũng được.

### Tính tay Example 1 của bài 256

| nhà `i` | `costs[i]` | `dp[i][0]` | `dp[i][1]` | `dp[i][2]` |
|---:|---|---|---|---|
| 0 | `[17, 2, 17]` | 17 | 2 | 17 |
| 1 | `[16, 16, 5]` | 16 + min(2, 17) = **18** | 16 + min(17, 17) = **33** | 5 + min(17, 2) = **7** |
| 2 | `[14, 3, 19]` | 14 + min(33, 7) = **21** | 3 + min(18, 7) = **10** | 19 + min(18, 33) = **37** |

Đáp án = `min(21, 10, 37) = 10`. Đúng như đề!

Để cài đặt ta cần một **bảng 2 chiều** `dp` kích thước `n x 3`. Có một cái bẫy kinh điển khi tạo bảng trong Python — xem ngay bên dưới.

### Kiến thức nền 5: tạo bảng 2 chiều đúng cách (và cái bẫy `[[0] * k] * n`)

Cách viết **trông** đúng nhưng **sai**:

```python
dp = [[0] * 3] * 2
```

Phép `* 2` ở ngoài **không tạo 2 hàng mới** — nó tạo **2 tham chiếu tới cùng một hàng**. Sửa `dp[0][0]` thì `dp[1][0]` cũng đổi theo. Hiện tượng này gọi là **aliasing** (hai tên cùng trỏ một đối tượng).

Cách đúng là **list comprehension**, để mỗi vòng lặp tạo một list mới:

```python
dp = [[0] * 3 for _ in range(2)]
```

(Bên trong `[0] * 3` thì không sao, vì `0` là số — số trong Python không thể bị sửa tại chỗ.)

In [14]:
bad = [[0] * 3] * 2
bad[0][0] = 99
print("Cách sai :", bad, "| hai hàng là một?", bad[0] is bad[1])

good = [[0] * 3 for _ in range(2)]
good[0][0] = 99
print("Cách đúng:", good, "| hai hàng là một?", good[0] is good[1])

Cách sai : [[99, 0, 0], [99, 0, 0]] | hai hàng là một? True
Cách đúng: [[99, 0, 0], [0, 0, 0]] | hai hàng là một? False


In [15]:
def paint_house_256(costs, verbose=False):
    """LeetCode 256 (đúng 3 màu): viết tay 3 công thức."""
    n = len(costs)
    dp = [[0] * 3 for _ in range(n)]
    dp[0] = costs[0][:]                        # [:] = sao chép, không sửa vào costs
    for i in range(1, n):
        dp[i][0] = costs[i][0] + min(dp[i - 1][1], dp[i - 1][2])
        dp[i][1] = costs[i][1] + min(dp[i - 1][0], dp[i - 1][2])
        dp[i][2] = costs[i][2] + min(dp[i - 1][0], dp[i - 1][1])
    if verbose:
        for i in range(n):
            print(f"nhà {i}: costs = {costs[i]} -> dp = {dp[i]}")
    return min(dp[n - 1])


costs_256 = [[17, 2, 17], [16, 16, 5], [14, 3, 19]]
print("Đáp án:", paint_house_256(costs_256, verbose=True))
assert paint_house_256(costs_256) == 10
assert paint_house_256([[7, 6, 2]]) == 2
assert paint_house_256([[1, 5, 3], [2, 9, 4]]) == 5     # Example 1 của 265 cũng có k = 3
print("LeetCode 256: PASS")

nhà 0: costs = [17, 2, 17] -> dp = [17, 2, 17]
nhà 1: costs = [16, 16, 5] -> dp = [18, 33, 7]
nhà 2: costs = [14, 3, 19] -> dp = [21, 10, 37]
Đáp án: 10
LeetCode 256: PASS


### Từ 3 màu lên `k` màu

Ba công thức của bài 256 thực ra là **một** công thức viết 3 lần:

```text
dp[i][c] = costs[i][c] + min( dp[i-1][c'] với mọi c' != c )
```

Với 3 màu, "mọi `c'` khác `c`" chỉ có 2 giá trị nên ta viết tay được `min(a, b)`. Với `k = 20` màu thì không ai viết tay 20 công thức, mỗi công thức min của 19 số — ta dùng **vòng lặp** và generator expression `min(prev[x] for x in range(k) if x != c)` đã học ở mục 2.

Liên hệ với Cách 2: `best(i, prev)` nhìn **về tương lai** (nhà `i` tới cuối), `dp[i][c]` nhìn **về quá khứ** (nhà 0 tới `i`). Hai cách định nghĩa đều đúng, đều có `O(nk)` trạng thái và `O(k)` việc mỗi trạng thái. Bảng DP từ trái sang tự nhiên hơn khi lặp bằng `for`, và là nền cho các tối ưu tiếp theo.

## 9. Cách 3 — DP bảng `dp[i][c]`, thời gian `O(nk²)`

Gom lại đầy đủ bốn thành phần của một lời giải DP:

| Thành phần | Nội dung |
|---|---|
| **Trạng thái** | `dp[i][c]` = chi phí nhỏ nhất tô nhà `0..i`, nhà `i` màu `c` |
| **Khởi tạo** | `dp[0][c] = costs[0][c]` với mọi `c` |
| **Chuyển trạng thái** | `dp[i][c] = costs[i][c] + min(dp[i-1][c'] với c' != c)` |
| **Thứ tự tính** | `i` tăng dần từ 1 tới `n - 1` (hàng `i` chỉ cần hàng `i - 1` đã xong) |
| **Đáp án** | `min(dp[n-1])` |

Hình dung: mỗi ô `dp[i][c]` "nhìn lên" hàng trên và lấy ô nhỏ nhất **không nằm cùng cột**.

```text
hàng i-1:  [ dp[i-1][0]  dp[i-1][1]  dp[i-1][2]  dp[i-1][3] ]
                 \           |  (cấm)    /           /
hàng i  :         ------> dp[i][1] <-----------------
```

In [16]:
def min_cost_dp_table(costs):
    """Cách 3: DP bảng n x k, mỗi ô lấy min của k-1 ô hàng trên. Thời gian O(n k^2)."""
    n, k = len(costs), len(costs[0])
    dp = [[0] * k for _ in range(n)]
    dp[0] = costs[0][:]
    for i in range(1, n):
        for c in range(k):
            dp[i][c] = costs[i][c] + min(dp[i - 1][x] for x in range(k) if x != c)
    return min(dp[n - 1])


for name, cs, expected in [("Example 1", costs_ex1, 5), ("Example 2", costs_ex2, 5),
                           ("costs_trap", costs_trap, 3), ("costs_demo", costs_demo, 5)]:
    result = min_cost_dp_table(cs)
    print(f"{name:<10}: {result} (mong đợi {expected})")
    assert result == expected

Example 1 : 5 (mong đợi 5)
Example 2 : 5 (mong đợi 5)
costs_trap: 3 (mong đợi 3)
costs_demo: 5 (mong đợi 5)


### Trace Cách 3 trên `costs_demo`

Hàng 0: `dp[0] = costs[0] = [3, 1, 4, 2]`. Ba hàng sau tính như sau (cột "các ô hàng trên được xét" đã bỏ ô cùng cột `c`):

| nhà `i` | màu `c` | `costs[i][c]` | các ô hàng trên được xét | min | `dp[i][c]` |
|---:|---:|---:|---|---:|---:|
| 1 | 0 | 2 | 1, 4, 2 | 1 | **3** |
| 1 | 1 | 7 | 3, 4, 2 | 2 | **9** |
| 1 | 2 | 1 | 3, 1, 2 | 1 | **2** |
| 1 | 3 | 8 | 3, 1, 4 | 1 | **9** |
| 2 | 0 | 5 | 9, 2, 9 | 2 | **7** |
| 2 | 1 | 2 | 3, 2, 9 | 2 | **4** |
| 2 | 2 | 6 | 3, 9, 9 | 3 | **9** |
| 2 | 3 | 2 | 3, 9, 2 | 2 | **4** |
| 3 | 0 | 4 | 4, 9, 4 | 4 | **8** |
| 3 | 1 | 1 | 7, 9, 4 | 4 | **5** |
| 3 | 2 | 3 | 7, 4, 4 | 4 | **7** |
| 3 | 3 | 5 | 7, 4, 9 | 4 | **9** |

Bảng cuối: `dp[1] = [3, 9, 2, 9]`, `dp[2] = [7, 4, 9, 4]`, `dp[3] = [8, 5, 7, 9]` → đáp án `min(dp[3]) = 5`. Hàm `trace_dp_table` in đúng các bước này.

In [17]:
def trace_dp_table(costs):
    n, k = len(costs), len(costs[0])
    prev = costs[0][:]
    print(f"nhà 0: dp = {prev}")
    for i in range(1, n):
        cur = []
        for c in range(k):
            others = [prev[x] for x in range(k) if x != c]
            cur.append(costs[i][c] + min(others))
            print(f"   c={c}: {costs[i][c]} + min{others} = {costs[i][c]} + {min(others)} = {cur[-1]}")
        print(f"nhà {i}: dp = {cur}")
        prev = cur
    print("Đáp án = min(dp hàng cuối) =", min(prev))
    return min(prev)


trace_dp_table(costs_demo)

nhà 0: dp = [3, 1, 4, 2]
   c=0: 2 + min[1, 4, 2] = 2 + 1 = 3
   c=1: 7 + min[3, 4, 2] = 7 + 2 = 9
   c=2: 1 + min[3, 1, 2] = 1 + 1 = 2
   c=3: 8 + min[3, 1, 4] = 8 + 1 = 9
nhà 1: dp = [3, 9, 2, 9]
   c=0: 5 + min[9, 2, 9] = 5 + 2 = 7
   c=1: 2 + min[3, 2, 9] = 2 + 2 = 4
   c=2: 6 + min[3, 9, 9] = 6 + 3 = 9
   c=3: 2 + min[3, 9, 2] = 2 + 2 = 4
nhà 2: dp = [7, 4, 9, 4]
   c=0: 4 + min[4, 9, 4] = 4 + 4 = 8
   c=1: 1 + min[7, 9, 4] = 1 + 4 = 5
   c=2: 3 + min[7, 4, 4] = 3 + 4 = 7
   c=3: 5 + min[7, 4, 9] = 5 + 4 = 9
nhà 3: dp = [8, 5, 7, 9]
Đáp án = min(dp hàng cuối) = 5


5

### Độ phức tạp của Cách 3 và nút thắt

- **Thời gian**: `n` hàng × `k` ô mỗi hàng × `k - 1` phép so sánh mỗi ô = `O(nk²)`.
- **Bộ nhớ**: bảng `n x k` → `O(nk)` (có thể giảm còn `O(k)` vì hàng `i` chỉ cần hàng `i - 1`).

Cách 3 **đã đủ để AC** với constraints của đề. Nhưng follow-up hỏi `O(nk)` — tức là mỗi ô chỉ được tốn `O(1)` thay vì `O(k)`. Liệu có làm được?

Hãy nhìn lại cột **min** trong bảng trace phía trên, theo từng hàng:

- Hàng 1: min = `1, 2, 1, 1` → chỉ có **2 giá trị khác nhau**: 1 và 2.
- Hàng 2: min = `2, 2, 3, 2` → chỉ có **2 giá trị**: 2 và 3.
- Hàng 3: min = `4, 4, 4, 4` → chỉ có **1 giá trị**: 4.

Ta tính `k` lần phép "min của `k - 1` số", nhưng kết quả **gần như lúc nào cũng giống nhau**! Đây là công việc lặp lại — và công việc lặp lại là thứ ta có thể tính **một lần** rồi dùng lại.

## 10. Bước đột phá: "min của hàng khi bỏ đi một vị trí" chỉ cần 2 giá trị nhỏ nhất

Tách riêng câu hỏi nhỏ ra khỏi bài toán. Cho một list `prev` và một vị trí `c`:

> `min_except(prev, c)` = giá trị nhỏ nhất của `prev` khi **bỏ đi** phần tử ở vị trí `c`.

Ví dụ `prev = [8, 3, 6, 5]`:

| `c` bị bỏ | còn lại | `min_except` |
|---:|---|---:|
| 0 | 3, 6, 5 | 3 |
| 1 | 8, 6, 5 | **5** |
| 2 | 8, 3, 5 | 3 |
| 3 | 8, 3, 6 | 3 |

Giá trị nhỏ nhất của cả hàng là `min1 = 3` ở vị trí `idx1 = 1`; giá trị nhỏ thứ hai là `min2 = 5`. Quan sát:

- Khi bỏ một vị trí **không phải** `idx1` → số 3 vẫn còn đó, không ai nhỏ hơn nó → kết quả là `min1 = 3`.
- Khi bỏ **đúng** `idx1` → số 3 biến mất, người "giỏi nhì" lên thay → kết quả là `min2 = 5`.

Cell dưới kiểm tra quan sát này trên vài hàng, kể cả hàng có **hoà** (hai giá trị nhỏ nhất bằng nhau).

In [18]:
def min_except_slow(prev, c):
    """Cách chậm, O(k): duyệt hết các vị trí khác c."""
    return min(prev[x] for x in range(len(prev)) if x != c)


for prev in [[8, 3, 6, 5], [4, 2, 2, 7], [3, 1, 4, 2]]:
    results = [min_except_slow(prev, c) for c in range(len(prev))]
    two_smallest_sorted = sorted(prev)[:2]
    print(f"prev = {prev}: min_except theo từng c = {results} | các giá trị xuất hiện = {sorted(set(results))}"
          f" | 2 số nhỏ nhất (sắp xếp) = {two_smallest_sorted}")

prev = [8, 3, 6, 5]: min_except theo từng c = [3, 5, 3, 3] | các giá trị xuất hiện = [3, 5] | 2 số nhỏ nhất (sắp xếp) = [3, 5]
prev = [4, 2, 2, 7]: min_except theo từng c = [2, 2, 2, 2] | các giá trị xuất hiện = [2] | 2 số nhỏ nhất (sắp xếp) = [2, 2]
prev = [3, 1, 4, 2]: min_except theo từng c = [1, 2, 1, 1] | các giá trị xuất hiện = [1, 2] | 2 số nhỏ nhất (sắp xếp) = [1, 2]


### Quy tắc vàng (và chuyện "hoà")

Gọi `min1` là giá trị nhỏ nhất của `prev`, `idx1` là **vị trí** của nó, `min2` là giá trị nhỏ nhất của **phần còn lại sau khi bỏ đúng phần tử ở `idx1`** (gọi là "nhỏ nhì"). Khi đó:

```text
min_except(prev, c) = min2   nếu c == idx1
                      min1   nếu c != idx1
```

**Chuyện hoà**: với `prev = [4, 2, 2, 7]`, có **hai** số 2. Ta chọn `idx1 = 1` (số 2 đầu tiên) thì `min2` là min của `[4, 2, 7]` = **2** (số 2 thứ hai) — chứ **không phải** 4. Nói cách khác, `min2` là "nhỏ nhì **tính cả lặp lại**", không phải "giá trị khác biệt nhỏ nhì". Nhờ vậy:

- bỏ vị trí 1 → còn số 2 ở vị trí 2 → kết quả 2 = `min2` ✓
- bỏ vị trí 2 → còn số 2 ở vị trí 1 → kết quả 2 = `min1` ✓

Đây đúng là điều xảy ra ở hàng `[3, 1, 4, 2]` → `[1, 2, 1, 1]` và ở hàng 3 của trace (`[7, 4, 9, 4]` → mọi min đều là 4).

**Hệ quả**: nếu biết trước bộ ba `(min1, min2, idx1)` của hàng trên, mỗi ô `dp[i][c]` chỉ tốn **`O(1)`**: một phép so sánh `c == idx1` và một phép cộng. Câu hỏi còn lại: tìm bộ ba đó thế nào cho nhanh? Sắp xếp tốn `O(k log k)` — ta làm được tốt hơn: **một lần duyệt `O(k)`**.

### Kiến thức nền 6: tìm 2 giá trị nhỏ nhất trong **một lần duyệt**

Hình dung một cuộc thi: ta giữ **huy chương vàng** (`min1`, kèm vị trí `idx1`) và **huy chương bạc** (`min2`). Ban đầu cả hai đều trống (`INF`). Lần lượt mỗi thí sinh `x` bước vào:

1. Nếu `x < min1` (phá kỷ lục vàng): **vàng cũ tụt xuống bạc** (`min2 = min1`), rồi `x` thành vàng (`min1 = x`, `idx1 = j`).
2. Ngược lại, nếu `x < min2` (chỉ thắng được bạc): `x` thành bạc (`min2 = x`).
3. Ngược lại: không làm gì.

Chú ý dùng `elif` ở bước 2: nếu `x` vừa lên vàng thì **không được** xét tiếp bước 2 cho chính `x`.

Trace tay với `[8, 3, 6, 5]`:

| bước `j` | `x` | xử lý | `min1` | `min2` | `idx1` |
|---|---:|---|---:|---:|---:|
| bắt đầu | | | inf | inf | -1 |
| 0 | 8 | 8 < inf → lên vàng, vàng cũ (inf) xuống bạc | 8 | inf | 0 |
| 1 | 3 | 3 < 8 → lên vàng, 8 xuống bạc | 3 | 8 | 1 |
| 2 | 6 | 6 ≥ 3 nhưng 6 < 8 → thay bạc | 3 | 6 | 1 |
| 3 | 5 | 5 ≥ 3 nhưng 5 < 6 → thay bạc | 3 | 5 | 1 |

Trace tay với hàng có hoà `[4, 2, 2, 7]`:

| bước `j` | `x` | xử lý | `min1` | `min2` | `idx1` |
|---|---:|---|---:|---:|---:|
| bắt đầu | | | inf | inf | -1 |
| 0 | 4 | 4 < inf → lên vàng | 4 | inf | 0 |
| 1 | 2 | 2 < 4 → lên vàng, 4 xuống bạc | 2 | 4 | 1 |
| 2 | 2 | 2 **không** < 2 (so sánh nghiêm ngặt), nhưng 2 < 4 → thay bạc | 2 | 2 | 1 |
| 3 | 7 | không thắng ai | 2 | 2 | 1 |

Kết quả `(2, 2, 1)` — đúng tinh thần "nhỏ nhì tính cả lặp lại" ở trên.

In [19]:
INF = float("inf")


def two_smallest(values):
    """Một lần duyệt O(k): trả về (min1, min2, idx1).
    min1 = nhỏ nhất, idx1 = vị trí của nó, min2 = nhỏ nhất trong phần còn lại (tính cả lặp lại)."""
    min1, min2, idx1 = INF, INF, -1
    for j, x in enumerate(values):
        if x < min1:
            min2 = min1                  # vàng cũ tụt xuống bạc
            min1, idx1 = x, j            # x lên vàng
        elif x < min2:
            min2 = x                     # x chỉ thắng bạc
    return min1, min2, idx1


def trace_two_smallest(values):
    min1, min2, idx1 = INF, INF, -1
    print(f"values = {values}")
    for j, x in enumerate(values):
        if x < min1:
            min2, min1, idx1 = min1, x, j
            action = "lên vàng (vàng cũ xuống bạc)"
        elif x < min2:
            min2 = x
            action = "thay bạc"
        else:
            action = "không đổi"
        print(f"  j={j}, x={x}: {action:<28} -> min1={min1}, min2={min2}, idx1={idx1}")
    return min1, min2, idx1


print("Kết quả:", trace_two_smallest([8, 3, 6, 5]))
print("Kết quả:", trace_two_smallest([4, 2, 2, 7]))

values = [8, 3, 6, 5]
  j=0, x=8: lên vàng (vàng cũ xuống bạc) -> min1=8, min2=inf, idx1=0
  j=1, x=3: lên vàng (vàng cũ xuống bạc) -> min1=3, min2=8, idx1=1
  j=2, x=6: thay bạc                     -> min1=3, min2=6, idx1=1
  j=3, x=5: thay bạc                     -> min1=3, min2=5, idx1=1
Kết quả: (3, 5, 1)
values = [4, 2, 2, 7]
  j=0, x=4: lên vàng (vàng cũ xuống bạc) -> min1=4, min2=inf, idx1=0
  j=1, x=2: lên vàng (vàng cũ xuống bạc) -> min1=2, min2=4, idx1=1
  j=2, x=2: thay bạc                     -> min1=2, min2=2, idx1=1
  j=3, x=7: không đổi                    -> min1=2, min2=2, idx1=1
Kết quả: (2, 2, 1)


In [20]:
def min_except_fast(prev, c):
    """Cách nhanh, dùng quy tắc vàng. (Ở đây tính lại two_smallest mỗi lần chỉ để minh hoạ;
    trong thuật toán thật ta tính bộ ba MỘT lần cho cả hàng.)"""
    min1, min2, idx1 = two_smallest(prev)
    return min2 if c == idx1 else min1


# Kiểm tra quy tắc vàng trên 2000 hàng ngẫu nhiên, giá trị nhỏ để có nhiều "hoà"
random.seed(42)
for _ in range(2000):
    k = random.randint(2, 6)
    prev = [random.randint(1, 4) for _ in range(k)]
    for c in range(k):
        assert min_except_fast(prev, c) == min_except_slow(prev, c), (prev, c)
    assert two_smallest(prev)[:2] == tuple(sorted(prev)[:2])
print("Quy tắc vàng đúng trên 2000 hàng ngẫu nhiên (kể cả có hoà)!")

Quy tắc vàng đúng trên 2000 hàng ngẫu nhiên (kể cả có hoà)!


## 11. Cách 4 — DP `O(nk)` với `min1`, `min2`, `idx1` (cách chính)

Ghép quy tắc vàng vào Cách 3. Mỗi hàng làm 2 việc, mỗi việc `O(k)`:

1. **Tóm tắt hàng trên** thành bộ ba `(min1, min2, idx1) = two_smallest(prev)`.
2. **Tính hàng mới**: với mỗi màu `c`,

```text
cur[c] = costs[i][c] + (min2 if c == idx1 else min1)
```

Sau đó `prev = cur` và sang nhà tiếp theo. Đáp án = `min(prev)` ở hàng cuối.

Mỗi hàng tốn `O(k) + O(k) = O(k)`, có `n` hàng → **`O(nk)`** — đúng yêu cầu follow-up. Bộ nhớ chỉ cần hàng hiện tại và hàng trước: `O(k)`.

Lưu ý đọc đúng ý nghĩa: `idx1` là **màu** mà phương án rẻ nhất tới nhà `i - 1` đang dùng. Nhà `i` mà định dùng đúng màu đó thì phải "lùi" xuống phương án rẻ nhì của nhà `i - 1`, vốn kết thúc bằng một màu khác.

In [21]:
def min_cost_two_min(costs):
    """Cách 4: DP O(nk) thời gian, O(k) bộ nhớ — mỗi hàng chỉ cần (min1, min2, idx1) của hàng trên."""
    n, k = len(costs), len(costs[0])
    prev = costs[0][:]                                   # dp của nhà 0
    for i in range(1, n):
        min1, min2, idx1 = two_smallest(prev)            # tóm tắt hàng trên: O(k)
        cur = [0] * k
        for c in range(k):                               # tính hàng mới: O(k)
            cur[c] = costs[i][c] + (min2 if c == idx1 else min1)
        prev = cur
    return min(prev)


for name, cs, expected in [("Example 1", costs_ex1, 5), ("Example 2", costs_ex2, 5),
                           ("costs_trap", costs_trap, 3), ("costs_demo", costs_demo, 5),
                           ("costs_256", costs_256, 10)]:
    result = min_cost_two_min(cs)
    print(f"{name:<10}: {result} (mong đợi {expected})")
    assert result == expected

Example 1 : 5 (mong đợi 5)
Example 2 : 5 (mong đợi 5)
costs_trap: 3 (mong đợi 3)
costs_demo: 5 (mong đợi 5)
costs_256 : 10 (mong đợi 10)


### Trace Cách 4 trên `costs_demo`

Ô in **đậm** là lúc màu `c` trùng `idx1` của hàng trên, buộc phải dùng `min2`.

| nhà `i` | `costs[i]` | bộ ba hàng trên `(min1, min2, idx1)` | tính từng màu `c` | `dp[i]` | bộ ba mới |
|---:|---|---|---|---|---|
| 0 | `[3, 1, 4, 2]` | — (nhà đầu) | `dp[0] = costs[0]` | `[3, 1, 4, 2]` | `(1, 2, 1)` |
| 1 | `[2, 7, 1, 8]` | `(1, 2, 1)` | c0: 2+1=3; **c1: 7+min2=7+2=9**; c2: 1+1=2; c3: 8+1=9 | `[3, 9, 2, 9]` | `(2, 3, 2)` |
| 2 | `[5, 2, 6, 2]` | `(2, 3, 2)` | c0: 5+2=7; c1: 2+2=4; **c2: 6+min2=6+3=9**; c3: 2+2=4 | `[7, 4, 9, 4]` | `(4, 4, 1)` ← hoà |
| 3 | `[4, 1, 3, 5]` | `(4, 4, 1)` | c0: 4+4=8; **c1: 1+min2=1+4=5**; c2: 3+4=7; c3: 5+4=9 | `[8, 5, 7, 9]` | `(5, 7, 1)` |

Đáp án = `min1` của hàng cuối = **5**. Các hàng `dp` khớp từng số với bảng của Cách 3 — nhưng mỗi ô giờ chỉ tốn một phép so sánh.

In [22]:
def trace_two_min(costs):
    n, k = len(costs), len(costs[0])
    prev = costs[0][:]
    print(f"nhà 0: costs={costs[0]} -> dp={prev} | (min1, min2, idx1) = {two_smallest(prev)}")
    for i in range(1, n):
        min1, min2, idx1 = two_smallest(prev)
        print(f"nhà {i}: costs={costs[i]}, hàng trên có min1={min1}, min2={min2}, idx1={idx1}")
        cur = []
        for c in range(k):
            if c == idx1:
                cur.append(costs[i][c] + min2)
                print(f"   c={c}: {costs[i][c]} + min2({min2}) = {cur[-1]}   <- c trùng idx1, phải dùng min2")
            else:
                cur.append(costs[i][c] + min1)
                print(f"   c={c}: {costs[i][c]} + min1({min1}) = {cur[-1]}")
        print(f"   -> dp={cur} | (min1, min2, idx1) = {two_smallest(cur)}")
        prev = cur
    print("Đáp án = min1 của hàng cuối =", two_smallest(prev)[0])
    return min(prev)


trace_two_min(costs_demo)

nhà 0: costs=[3, 1, 4, 2] -> dp=[3, 1, 4, 2] | (min1, min2, idx1) = (1, 2, 1)
nhà 1: costs=[2, 7, 1, 8], hàng trên có min1=1, min2=2, idx1=1
   c=0: 2 + min1(1) = 3
   c=1: 7 + min2(2) = 9   <- c trùng idx1, phải dùng min2
   c=2: 1 + min1(1) = 2
   c=3: 8 + min1(1) = 9
   -> dp=[3, 9, 2, 9] | (min1, min2, idx1) = (2, 3, 2)
nhà 2: costs=[5, 2, 6, 2], hàng trên có min1=2, min2=3, idx1=2
   c=0: 5 + min1(2) = 7
   c=1: 2 + min1(2) = 4
   c=2: 6 + min2(3) = 9   <- c trùng idx1, phải dùng min2
   c=3: 2 + min1(2) = 4
   -> dp=[7, 4, 9, 4] | (min1, min2, idx1) = (4, 4, 1)
nhà 3: costs=[4, 1, 3, 5], hàng trên có min1=4, min2=4, idx1=1
   c=0: 4 + min1(4) = 8
   c=1: 1 + min2(4) = 5   <- c trùng idx1, phải dùng min2
   c=2: 3 + min1(4) = 7
   c=3: 5 + min1(4) = 9
   -> dp=[8, 5, 7, 9] | (min1, min2, idx1) = (5, 7, 1)
Đáp án = min1 của hàng cuối = 5


5

### Nhận xét về trace

- **Nhà 1, màu 1**: phương án rẻ nhất tới nhà 0 là màu 1 (giá 1). Nhà 1 muốn màu 1 thì nhà 0 phải đổi sang phương án rẻ nhì (màu 3, giá 2) → `7 + 2 = 9`.
- **Nhà 3, màu 1**: hàng trên `[7, 4, 9, 4]` có **hoà** ở giá trị 4 (màu 1 và màu 3). `idx1 = 1` nên màu 1 dùng `min2 = 4` — chính là số 4 của **màu 3**. Kết quả `1 + 4 = 5` là đáp án cuối cùng. Nếu ta hiểu nhầm `min2` là "giá trị khác biệt nhỏ nhì" (tức 7) thì sẽ ra `1 + 7 = 8` — sai!
- **Lần ngược ra cách tô**: nhà 3 rẻ nhất ở màu 1 (`dp = 5`). Nhà 2 phải khác màu 1, rẻ nhất là màu 3 (`dp = 4`). Nhà 1 phải khác màu 3, rẻ nhất là màu 2 (`dp = 2`). Nhà 0 phải khác màu 2, rẻ nhất là màu 1 (`dp = 1`). Cách tô `[1, 2, 3, 1]`, tổng `1 + 1 + 2 + 1 = 5` ✓. (Bài tập 5 sẽ yêu cầu bạn tự cài đặt việc lần ngược này.)

## 12. Vì sao chỉ cần 2 giá trị nhỏ nhất?

**Mệnh đề.** Với mọi màu `c`: `min( prev[x] với x != c ) = min2 nếu c == idx1, ngược lại = min1`.

**Chứng minh bằng lời**, chia 2 trường hợp:

1. **`c != idx1`**: phần tử ở vị trí `idx1` (giá trị `min1`) **vẫn còn** trong tập đang xét. Không phần tử nào của `prev` nhỏ hơn `min1` (vì `min1` là nhỏ nhất của cả hàng). Vậy min của tập là đúng `min1`.
2. **`c == idx1`**: ta bỏ **đúng** phần tử ở vị trí `idx1` và giữ lại tất cả phần tử khác. Theo định nghĩa, `min2` chính là giá trị nhỏ nhất của "tất cả phần tử khác `idx1`". Vậy kết quả là `min2`.

Hai trường hợp phủ hết mọi `c`, nên mệnh đề đúng — kể cả khi có hoà.

**Tại sao 1 giá trị là không đủ?** Nếu chỉ nhớ `min1`, ta không biết trả lời gì khi `c == idx1`. Ví dụ hàng `[8, 3, 6, 5]`, bỏ vị trí 1 thì đáp án là 5 — con số này không suy ra được từ `min1 = 3`.

**Tại sao không bao giờ cần giá trị thứ 3?** Vì mỗi ô chỉ **cấm đúng một** màu. Bỏ đi một phần tử thì trong "top 2" luôn còn lại ít nhất một phần tử. Tổng quát: nếu mỗi ô cấm `t` màu, ta cần nhớ `t + 1` giá trị nhỏ nhất (kèm vị trí).

**Bất biến (invariant) của Cách 4**: ngay trước khi xử lý nhà `i`, `prev[c]` = chi phí nhỏ nhất tô nhà `0..i-1` với nhà `i-1` màu `c`, và `(min1, min2, idx1)` là "top 2" đúng của `prev`. Khởi tạo đúng (nhà 0), mỗi bước giữ đúng (theo công thức DP + mệnh đề trên), nên khi kết thúc `min(prev)` là đáp án.

**Độ phức tạp Cách 4**: thời gian `O(nk)` (mỗi hàng 2 lần duyệt `k` phần tử), bộ nhớ `O(k)` (hai list độ dài `k`).

## 13. Cách 5 — chỉ giữ 3 biến: bộ nhớ thêm `O(1)`

Nhìn kỹ Cách 4: list `cur` được tạo ra **chỉ để** ngay sau đó tính `two_smallest(cur)`. Ta có thể đưa từng giá trị `cur[c]` **thẳng vào** "cuộc thi huy chương" ngay khi vừa tính xong, không cần lưu cả list!

Vậy với mỗi nhà ta chỉ cần:

- **3 biến của nhà trước**: `prev_min1`, `prev_min2`, `prev_idx1` (dùng để tính chi phí từng màu).
- **3 biến đang xây cho nhà hiện tại**: `cur_min1`, `cur_min2`, `cur_idx1` (cập nhật như `two_smallest`).

Hết vòng màu thì "chuyển giao": `prev_* = cur_*`. Bộ nhớ thêm là `O(1)` — không phụ thuộc `n` hay `k` (không tính input `costs`).

**Mẹo "nhà ảo"**: thay vì xử lý nhà 0 riêng, ta tưởng tượng có một **nhà ảo** đứng trước nhà 0, chi phí 0 và **không có màu**: `prev_min1 = prev_min2 = 0`, `prev_idx1 = -1`. Vì không màu `c` nào bằng `-1`, mọi màu của nhà 0 đều nhận `cost + 0` — đúng bằng `dp[0][c] = costs[0][c]`. Nhờ vậy vòng lặp xử lý mọi nhà giống hệt nhau, code ngắn và ít lỗi biên hơn.

Một cách `O(1)` khác bạn có thể gặp trên mạng là **ghi đè luôn vào `costs`** (biến `costs[i][c]` thành `dp[i][c]`). Cách đó cũng `O(1)` bộ nhớ thêm nhưng **làm hỏng input** của người gọi — trong phỏng vấn nên hỏi trước có được sửa input không. Cách 5 dưới đây không sửa `costs`.

In [23]:
def min_cost_o1(costs):
    """Cách 5: O(nk) thời gian, O(1) bộ nhớ thêm — chỉ giữ (min1, min2, idx1) của nhà trước."""
    prev_min1, prev_min2, prev_idx1 = 0, 0, -1          # "nhà ảo" trước nhà 0: chi phí 0, không màu
    for row in costs:
        cur_min1, cur_min2, cur_idx1 = INF, INF, -1
        for c, cost in enumerate(row):
            total = cost + (prev_min2 if c == prev_idx1 else prev_min1)   # = dp[i][c]
            if total < cur_min1:                         # cập nhật "huy chương" ngay lập tức
                cur_min2 = cur_min1
                cur_min1, cur_idx1 = total, c
            elif total < cur_min2:
                cur_min2 = total
        prev_min1, prev_min2, prev_idx1 = cur_min1, cur_min2, cur_idx1   # chuyển giao SAU vòng màu
    return prev_min1


for name, cs, expected in [("Example 1", costs_ex1, 5), ("Example 2", costs_ex2, 5),
                           ("costs_trap", costs_trap, 3), ("costs_demo", costs_demo, 5),
                           ("costs_256", costs_256, 10)]:
    result = min_cost_o1(cs)
    print(f"{name:<10}: {result} (mong đợi {expected})")
    assert result == expected

Example 1 : 5 (mong đợi 5)
Example 2 : 5 (mong đợi 5)
costs_trap: 3 (mong đợi 3)
costs_demo: 5 (mong đợi 5)
costs_256 : 10 (mong đợi 10)


In [24]:
def trace_o1(costs):
    prev_min1, prev_min2, prev_idx1 = 0, 0, -1
    print(f"nhà ảo   : (min1, min2, idx1) = ({prev_min1}, {prev_min2}, {prev_idx1})")
    for i, row in enumerate(costs):
        cur_min1, cur_min2, cur_idx1 = INF, INF, -1
        totals = []
        for c, cost in enumerate(row):
            total = cost + (prev_min2 if c == prev_idx1 else prev_min1)
            totals.append(total)
            if total < cur_min1:
                cur_min2, cur_min1, cur_idx1 = cur_min1, total, c
            elif total < cur_min2:
                cur_min2 = total
        prev_min1, prev_min2, prev_idx1 = cur_min1, cur_min2, cur_idx1
        print(f"nhà {i}    : costs={row}, các total={totals} -> (min1, min2, idx1) = "
              f"({prev_min1}, {prev_min2}, {prev_idx1})")
    print("Đáp án = min1 cuối cùng =", prev_min1)
    return prev_min1


trace_o1(costs_demo)

nhà ảo   : (min1, min2, idx1) = (0, 0, -1)
nhà 0    : costs=[3, 1, 4, 2], các total=[3, 1, 4, 2] -> (min1, min2, idx1) = (1, 2, 1)
nhà 1    : costs=[2, 7, 1, 8], các total=[3, 9, 2, 9] -> (min1, min2, idx1) = (2, 3, 2)
nhà 2    : costs=[5, 2, 6, 2], các total=[7, 4, 9, 4] -> (min1, min2, idx1) = (4, 4, 1)
nhà 3    : costs=[4, 1, 3, 5], các total=[8, 5, 7, 9] -> (min1, min2, idx1) = (5, 7, 1)
Đáp án = min1 cuối cùng = 5


5

## 14. Code nộp LeetCode

Ta nộp **Cách 5** (thời gian `O(nk)`, bộ nhớ thêm `O(1)`, không sửa input). Vài điểm khi chuyển từ hàm demo sang `class Solution`:

- Giữ **đúng tên method** `minCostII` và signature `(self, costs: List[List[int]]) -> int`. Trên LeetCode, `List` đã được import sẵn.
- Không `print` gì trong code nộp.
- Dùng `float("inf")` cho giá trị khởi đầu của `cur_min1`, `cur_min2`. Vì `k >= 2` nên `cur_min2` luôn được cập nhật thành số hữu hạn sau mỗi hàng.
- (Ngoài lề) Nếu đề cho phép `k = 1` thì với `n >= 2` không có cách tô hợp lệ, hàm sẽ trả `inf`. Constraints của đề loại trường hợp này, nên ta không cần xử lý.

Nếu đi phỏng vấn mà bí, nộp Cách 3 (`O(nk²)`) cũng AC với constraints này — nhưng hãy chủ động trình bày được Cách 4/5 cho câu follow-up.

In [25]:
class Solution:
    def minCostII(self, costs: List[List[int]]) -> int:
        prev_min1, prev_min2, prev_idx1 = 0, 0, -1      # nhà ảo trước nhà 0
        for row in costs:
            cur_min1 = cur_min2 = float("inf")
            cur_idx1 = -1
            for c, cost in enumerate(row):
                total = cost + (prev_min2 if c == prev_idx1 else prev_min1)
                if total < cur_min1:
                    cur_min1, cur_min2, cur_idx1 = total, cur_min1, c
                elif total < cur_min2:
                    cur_min2 = total
            prev_min1, prev_min2, prev_idx1 = cur_min1, cur_min2, cur_idx1
        return prev_min1

In [26]:
sol = Solution()
tests = [
    ([[1, 5, 3], [2, 9, 4]], 5, "Example 1"),
    ([[1, 3], [2, 4]], 5, "Example 2"),
    ([[7, 3, 9]], 3, "n = 1: chỉ chọn màu rẻ nhất"),
    ([[5, 2], [9, 1]], 6, "k = 2: bắt buộc xen kẽ màu"),
    ([[1, 100], [1, 100], [1, 100]], 102, "k = 2, 3 nhà: 0,1,0"),
    ([[1, 2, 30], [1, 30, 30]], 3, "bẫy tham lam"),
    (costs_demo, 5, "costs_demo (có hoà)"),
    ([[1, 1], [1, 1], [1, 1]], 3, "mọi giá bằng nhau"),
    ([[20] * 20 for _ in range(100)], 2000, "kích thước tối đa, mọi giá = 20"),
    ([[1] * 20 for _ in range(100)], 100, "kích thước tối đa, mọi giá = 1"),
]
for costs, expected, name in tests:
    snapshot = [row[:] for row in costs]
    result = sol.minCostII(costs)
    print(f"{name:<36} -> {result:>5} (mong đợi {expected})")
    assert result == expected, (name, result, expected)
    assert costs == snapshot, "không được sửa input!"
print("Tất cả test của class Solution đều PASS!")

Example 1                            ->     5 (mong đợi 5)
Example 2                            ->     5 (mong đợi 5)
n = 1: chỉ chọn màu rẻ nhất          ->     3 (mong đợi 3)
k = 2: bắt buộc xen kẽ màu           ->     6 (mong đợi 6)
k = 2, 3 nhà: 0,1,0                  ->   102 (mong đợi 102)
bẫy tham lam                         ->     3 (mong đợi 3)
costs_demo (có hoà)                  ->     5 (mong đợi 5)
mọi giá bằng nhau                    ->     3 (mong đợi 3)
kích thước tối đa, mọi giá = 20      ->  2000 (mong đợi 2000)
kích thước tối đa, mọi giá = 1       ->   100 (mong đợi 100)
Tất cả test của class Solution đều PASS!


### Kiểm thử đối chiếu chéo tất cả các cách

Test bằng tay chỉ phủ được vài tình huống ta nghĩ ra. Để chắc chắn hơn, ta sinh **hàng trăm input ngẫu nhiên nhỏ** (với `random.seed` cố định để lần nào chạy cũng ra cùng dữ liệu) và yêu cầu **mọi cách đều khớp brute force** — "trọng tài" không thể sai.

Với input lớn đúng cỡ constraints (`n = 100`, `k = 20`) thì brute force bó tay, nên ta đối chiếu các cách nhanh với nhau. Ta cũng cố ý dùng chi phí trong khoảng nhỏ (1 tới 3) ở một số lượt để tạo thật nhiều **hoà** — nơi lỗi `min2` hay ẩn nấp.

In [27]:
random.seed(265)
methods = {
    "memo": lambda cs: min_cost_memo(cs)[0],
    "dp_table": min_cost_dp_table,
    "two_min": min_cost_two_min,
    "o1": min_cost_o1,
    "Solution": Solution().minCostII,
}

# 1) Input nhỏ: so với brute force
for trial in range(400):
    n, k = random.randint(1, 6), random.randint(2, 4)
    hi = 3 if trial % 2 == 0 else 20                     # nửa số lượt có giá trị nhỏ -> nhiều hoà
    cs = [[random.randint(1, hi) for _ in range(k)] for _ in range(n)]
    expected = min_cost_bruteforce(cs)
    for name, f in methods.items():
        assert f(cs) == expected, (name, cs, f(cs), expected)
print("400 input nhỏ: mọi cách khớp brute force")

# 2) Input cỡ tối đa: các cách nhanh khớp nhau
for trial in range(30):
    cs = [[random.randint(1, 20) for _ in range(20)] for _ in range(100)]
    answers = {name: f(cs) for name, f in methods.items()}
    assert len(set(answers.values())) == 1, answers
print("30 input n = 100, k = 20: mọi cách khớp nhau, ví dụ đáp án lượt cuối =", answers["o1"])

400 input nhỏ: mọi cách khớp brute force


30 input n = 100, k = 20: mọi cách khớp nhau, ví dụ đáp án lượt cuối = 160


In [28]:
import time

# Đếm số lần chạy vòng lặp trong cùng (thao tác cơ bản) theo công thức
print(f"{'n':>5} {'k':>5} | Cách 3 ~ n*k*(k-1) | Cách 4/5 ~ 2*n*k")
for n, k in [(100, 20), (100, 200), (1000, 1000)]:
    print(f"{n:>5} {k:>5} | {n * k * (k - 1):>18,} | {2 * n * k:>13,}")

# Đo thời gian thật trên một input k lớn hơn constraints (chỉ để thấy khác biệt)
random.seed(7)
big = [[random.randint(1, 20) for _ in range(150)] for _ in range(60)]
for name, f in [("Cách 3 dp_table", min_cost_dp_table), ("Cách 5 o1", min_cost_o1)]:
    t0 = time.perf_counter()
    ans = f(big)
    print(f"{name:<16}: đáp án {ans}, mất {time.perf_counter() - t0:.4f} giây")

    n     k | Cách 3 ~ n*k*(k-1) | Cách 4/5 ~ 2*n*k
  100    20 |             38,000 |         4,000
  100   200 |          3,980,000 |        40,000
 1000  1000 |        999,000,000 |     2,000,000
Cách 3 dp_table : đáp án 60, mất 0.0787 giây
Cách 5 o1       : đáp án 60, mất 0.0005 giây


## 15. Vì sao đúng? Tổng hợp bất biến và độ phức tạp

**1. Cấu trúc con tối ưu (optimal substructure).** Xét một cách tô **tối ưu** cho các nhà `0..i` với nhà `i` màu `c`. Nhà `i - 1` của nó có màu `c' != c`, và **phần tô các nhà `0..i-1` cũng phải tối ưu** trong số các cách kết thúc bằng màu `c'` — nếu không, thay nó bằng phần rẻ hơn sẽ được cách tô rẻ hơn mà vẫn hợp lệ (vì ràng buộc chỉ liên quan nhà `i - 1` và `i`, và màu nhà `i - 1` không đổi). Từ đó:

- `dp[i][c] >= costs[i][c] + min(dp[i-1][c'] với c' != c)` (mọi cách tô đều phải đi qua một `c'` nào đó),
- và giá trị này **đạt được** bằng cách nối nhà `i` màu `c` vào phương án tốt nhất của `c'` tương ứng.

Nên công thức truy hồi cho **đúng** `dp[i][c]`. Quy nạp theo `i` (bắt đầu từ `dp[0][c] = costs[0][c]`) cho thấy cả bảng đúng; đáp án là `min(dp[n-1])` vì nhà cuối được phép mang bất kỳ màu nào.

**2. Quy tắc vàng** (mục 12) thay "min của `k - 1` số" bằng tra `min1`/`min2` mà **không làm thay đổi giá trị** — nên Cách 4 tính ra **đúng bảng** của Cách 3, chỉ nhanh hơn.

**3. Cách 5** tính đúng các giá trị `dp[i][c]` như Cách 4, chỉ khác là không lưu chúng lại; bộ ba `(min1, min2, idx1)` sau mỗi hàng đúng bằng `two_smallest(dp[i])`. Nhà ảo `(0, 0, -1)` cho ra `dp[0][c] = costs[0][c]`.

**Độ phức tạp** (`n` = số nhà, `k` = số màu):

| Cách | Thời gian | Bộ nhớ thêm |
|---|---|---|
| Tham lam | `O(nk)` | `O(n)` |
| 1. Brute force | `O(n · k^n)` | `O(n)` |
| 2a. Đệ quy thuần | `O(k · (k-1)^(n-1))` lời gọi | `O(n)` (call stack) |
| 2b. Đệ quy + memo | `O(nk²)` | `O(nk)` memo + `O(n)` call stack |
| 3. DP bảng | `O(nk²)` | `O(nk)` (hoặc `O(k)` nếu chỉ giữ 2 hàng) |
| 4. DP + top-2 | `O(nk)` | `O(k)` |
| 5. Ba biến | `O(nk)` | `O(1)` |

`O(nk)` là **tối ưu**: chỉ riêng việc đọc hết input đã tốn `n · k` bước, và giá ở ô nào cũng có thể thay đổi đáp án nên không thể bỏ qua ô nào.

## 16. So sánh các cách

| Cách | Ý tưởng | Đúng? | Thời gian | Bộ nhớ thêm | Khi nào dùng |
|---|---|---|---|---|---|
| Tham lam | mỗi nhà chọn màu rẻ nhất khác nhà trước | **Sai** | `O(nk)` | `O(n)` | Không dùng; chỉ để hiểu vì sao cần DP |
| 1. Brute force | thử đủ `k^n` cách tô | Đúng | `O(n · k^n)` | `O(n)` | Trọng tài kiểm thử với input nhỏ |
| 2. Đệ quy + memo | `best(i, prev)` từ nhà `i` tới cuối | Đúng | `O(nk²)` | `O(nk)` | Khi mới tìm ra công thức, nghĩ top-down dễ hơn |
| 3. DP bảng | `dp[i][c] = costs[i][c] + min(hàng trên trừ cột c)` | Đúng | `O(nk²)` | `O(nk)` | Bản dễ viết, đủ AC với constraints |
| 4. DP + top-2 | chỉ cần `min1, min2, idx1` của hàng trên | Đúng | `O(nk)` | `O(k)` | **Cách chính**, trả lời follow-up |
| 5. Ba biến | cập nhật top-2 ngay khi tính, nhà ảo | Đúng | `O(nk)` | `O(1)` | Bản nộp gọn nhất |

Con đường tư duy: **tham lam sai** → **thử hết** (đúng nhưng mũ) → nhận ra chỉ cần nhớ "màu nhà trước" (**trạng thái DP**) → **bảng** → nhận ra phép min bị lặp lại, chỉ cần **top-2** → không cần lưu hàng, chỉ cần **3 biến**.

## 17. Lỗi người mới hay gặp

### Lỗi 1: Dùng tham lam
Chọn màu rẻ nhất cho từng nhà rồi né trùng. Đúng cả 2 ví dụ đề nhưng sai với `[[1, 2, 30], [1, 30, 30]]` (ra 31 thay vì 3). **Sửa:** dùng DP, giữ mọi khả năng màu của nhà trước.

### Lỗi 2: Quên loại màu `c` khi lấy min hàng trên
```python
cur[c] = costs[i][c] + min(prev)          # SAI: có thể lấy đúng màu c của nhà trước
```
Kết quả cho phép hai nhà kề cùng màu → nhỏ hơn đáp án thật. **Sửa:** `min2 if c == idx1 else min1` (hoặc min trên `x != c`).

### Lỗi 3: Tóm tắt **chi phí thô** `costs[i-1]` thay vì **hàng dp** `prev`
```python
min1, min2, idx1 = two_smallest(costs[i - 1])   # SAI: quên chi phí tích luỹ của các nhà trước
```
**Sửa:** luôn tóm tắt hàng `dp` (tổng chi phí từ nhà 0 tới nhà `i - 1`).

### Lỗi 4: Quên cho "vàng cũ tụt xuống bạc"
```python
if x < min1:
    min1, idx1 = x, j        # SAI: thiếu min2 = min1
elif x < min2:
    min2 = x
```
Với hàng `[5, 2]`: ra `min2 = inf` thay vì 5. **Sửa:** `min2 = min1` **trước** khi gán `min1 = x`.

### Lỗi 5: Dùng `if` thay cho `elif`
```python
if x < min1: min2 = min1; min1 = x; idx1 = j
if x < min2: min2 = x        # SAI: x vừa lên vàng lại chiếm luôn ghế bạc
```
Với hàng `[5, 2]`: ra `(2, 2, 1)` thay vì `(2, 5, 1)` → có thể cho phép hai nhà kề cùng màu. **Sửa:** dùng `elif`.

### Lỗi 6: Khởi tạo `min1`, `min2` bằng 0
Mọi chi phí đều `>= 1` nên không số nào nhỏ hơn 0 → bộ ba mãi là `(0, 0, -1)` → kết quả chỉ là min của hàng cuối. **Sửa:** khởi tạo `INF` (riêng "nhà ảo" `0, 0, -1` của Cách 5 là bộ ba **của nhà trước**, khác hoàn toàn bộ ba đang xây).

### Lỗi 7: Ghi đè bộ ba của nhà trước khi vẫn đang dùng
Trong Cách 5, nếu gán `prev_min1, prev_min2, prev_idx1 = cur_...` **bên trong** vòng `for c`, các màu sau sẽ dùng thông tin **trộn** giữa nhà cũ và nhà mới. **Sửa:** chỉ chuyển giao **sau** khi duyệt xong mọi màu.

### Lỗi 8: Trả về `dp[-1][-1]` hoặc `dp[-1][0]`
Đáp án là `min(dp[-1])` — nhà cuối được mang bất kỳ màu nào. `dp[-1][-1]` chỉ là phương án "nhà cuối màu `k - 1`".

### Lỗi 9: Tạo bảng bằng `[[0] * k] * n`
Các hàng là cùng một list (aliasing, mục "Kiến thức nền 5") → sửa một hàng là sửa tất cả. **Sửa:** `[[0] * k for _ in range(n)]`.

Cell dưới chạy các phiên bản sai để bạn **tận mắt thấy** chúng sai.

In [29]:
def wrong_no_exclusion(costs):                       # Lỗi 2
    prev = costs[0][:]
    for i in range(1, len(costs)):
        m = min(prev)
        prev = [costs[i][c] + m for c in range(len(prev))]
    return min(prev)


def wrong_raw_costs(costs):                          # Lỗi 3
    prev = costs[0][:]
    for i in range(1, len(costs)):
        min1, min2, idx1 = two_smallest(costs[i - 1])
        prev = [costs[i][c] + (min2 if c == idx1 else min1) for c in range(len(prev))]
    return min(prev)


def make_wrong_two_min(top2):                        # dùng cho Lỗi 4, 5, 6
    def solve(costs):
        prev = costs[0][:]
        for i in range(1, len(costs)):
            min1, min2, idx1 = top2(prev)
            prev = [costs[i][c] + (min2 if c == idx1 else min1) for c in range(len(prev))]
        return min(prev)
    return solve


def top2_no_shift(values):                           # Lỗi 4
    min1, min2, idx1 = INF, INF, -1
    for j, x in enumerate(values):
        if x < min1:
            min1, idx1 = x, j
        elif x < min2:
            min2 = x
    return min1, min2, idx1


def top2_if_if(values):                              # Lỗi 5
    min1, min2, idx1 = INF, INF, -1
    for j, x in enumerate(values):
        if x < min1:
            min2, min1, idx1 = min1, x, j
        if x < min2:
            min2 = x
    return min1, min2, idx1


def top2_init_zero(values):                          # Lỗi 6
    min1, min2, idx1 = 0, 0, -1
    for j, x in enumerate(values):
        if x < min1:
            min2, min1, idx1 = min1, x, j
        elif x < min2:
            min2 = x
    return min1, min2, idx1


def wrong_overwrite_prev(costs):                     # Lỗi 7
    prev_min1, prev_min2, prev_idx1 = 0, 0, -1
    for row in costs:
        cur_min1, cur_min2, cur_idx1 = INF, INF, -1
        for c, cost in enumerate(row):
            total = cost + (prev_min2 if c == prev_idx1 else prev_min1)
            if total < cur_min1:
                cur_min2, cur_min1, cur_idx1 = cur_min1, total, c
            elif total < cur_min2:
                cur_min2 = total
            prev_min1, prev_min2, prev_idx1 = cur_min1, cur_min2, cur_idx1   # SAI: bên trong vòng c
    return prev_min1


def wrong_last_cell(costs):                          # Lỗi 8
    prev = costs[0][:]
    for i in range(1, len(costs)):
        min1, min2, idx1 = two_smallest(prev)
        prev = [costs[i][c] + (min2 if c == idx1 else min1) for c in range(len(prev))]
    return prev[-1]


print("top2_no_shift([5, 2]) =", top2_no_shift([5, 2]), "| đúng:", two_smallest([5, 2]))
print("top2_if_if([5, 2])    =", top2_if_if([5, 2]), "| đúng:", two_smallest([5, 2]))
cases = [
    ("Lỗi 2 quên loại màu c", wrong_no_exclusion, costs_ex1),
    ("Lỗi 3 dùng costs thô", wrong_raw_costs, costs_demo),
    ("Lỗi 4 quên dời xuống bạc", make_wrong_two_min(top2_no_shift), [[5, 2], [9, 1]]),
    ("Lỗi 5 if thay elif", make_wrong_two_min(top2_if_if), [[5, 2], [9, 1]]),
    ("Lỗi 6 khởi tạo bằng 0", make_wrong_two_min(top2_init_zero), costs_ex1),
    ("Lỗi 7 ghi đè bộ ba sớm", wrong_overwrite_prev, costs_demo),
    ("Lỗi 8 trả dp[-1][-1]", wrong_last_cell, costs_demo),
]
for name, f, cs in cases:
    print(f"{name:<26}: ra {f(cs):>3} | đúng là {min_cost_o1(cs)}  (input {cs})")

top2_no_shift([5, 2]) = (2, inf, 1) | đúng: (2, 5, 1)
top2_if_if([5, 2])    = (2, 2, 1) | đúng: (2, 5, 1)
Lỗi 2 quên loại màu c     : ra   3 | đúng là 5  (input [[1, 5, 3], [2, 9, 4]])
Lỗi 3 dùng costs thô      : ra   3 | đúng là 5  (input [[3, 1, 4, 2], [2, 7, 1, 8], [5, 2, 6, 2], [4, 1, 3, 5]])
Lỗi 4 quên dời xuống bạc  : ra  11 | đúng là 6  (input [[5, 2], [9, 1]])
Lỗi 5 if thay elif        : ra   3 | đúng là 6  (input [[5, 2], [9, 1]])
Lỗi 6 khởi tạo bằng 0     : ra   2 | đúng là 5  (input [[1, 5, 3], [2, 9, 4]])
Lỗi 7 ghi đè bộ ba sớm    : ra  18 | đúng là 5  (input [[3, 1, 4, 2], [2, 7, 1, 8], [5, 2, 6, 2], [4, 1, 3, 5]])
Lỗi 8 trả dp[-1][-1]      : ra   9 | đúng là 5  (input [[3, 1, 4, 2], [2, 7, 1, 8], [5, 2, 6, 2], [4, 1, 3, 5]])


## 18. Mẹo nhận dạng pattern

1. **"Mỗi vị trí chọn 1 trong `k` lựa chọn, ràng buộc chỉ giữa hai vị trí kề nhau, tối ưu tổng"** → DP với trạng thái **(vị trí, lựa chọn cuối cùng)**: `dp[i][c]`. Gặp lại ở Paint House, House Robber (lựa chọn = lấy / không lấy), Minimum Falling Path Sum...
2. **Tham lam trông hợp lý nhưng một lựa chọn "khoá" lựa chọn của bước sau** → thử tìm phản ví dụ 2 phần tử (như `costs_trap`). Nếu có, chuyển sang DP.
3. **Công thức có dạng "min / max của mọi phần tử trừ một phần tử"** → nhớ **top-2** (giá trị nhỏ nhất + vị trí, giá trị nhỏ nhì), hoặc mảng **min tiền tố / min hậu tố**. Biến `O(k)` mỗi ô thành `O(1)`.
4. **Follow-up đòi bỏ một thừa số `k`** → tìm phép tính **lặp lại** trong vòng trong (ở đây: min hàng trên bị tính lại `k` lần) và **tính trước một lần cho cả hàng**.
5. **Hàng `i` chỉ phụ thuộc hàng `i - 1`** → "cuộn mảng" (rolling array) chỉ giữ 1–2 hàng; nếu hàng chỉ được dùng qua một bản tóm tắt cỡ cố định (bộ ba) → giữ đúng bản tóm tắt đó, bộ nhớ `O(1)`.
6. **Mẹo "phần tử ảo"** (nhà ảo chi phí 0, không màu) giúp bỏ code xử lý riêng cho phần tử đầu.

## 19. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| **256. Paint House** (Medium, Premium) | Trường hợp `k = 3` — đã ôn ở mục 8 |
| **1289. Minimum Falling Path Sum II** (Hard) | **Y hệt** bài này: lưới `n x n`, mỗi hàng chọn 1 ô, hai hàng kề không cùng cột, giá trị có thể **âm** → Bài tập 6 |
| **931. Minimum Falling Path Sum** (Medium) | Ràng buộc ngược lại: hàng dưới chỉ được chọn cột `j-1, j, j+1` |
| **276. Paint Fence** (Medium, Premium) | **Đếm** số cách tô, không quá 2 cột liền nhau cùng màu |
| **1473. Paint House III** (Hard) | Thêm điều kiện số "khu phố" (neighborhood) = `target` → DP 3 chiều |
| **198. House Robber** (Medium) | Cùng tư duy "trạng thái = lựa chọn ở vị trí trước" |
| **1937. Maximum Number of Points with Cost** (Medium) | Dùng **max tiền tố / hậu tố** để giảm `O(k²)` → `O(k)` mỗi hàng — họ hàng của mẹo top-2 |

Hai biến thể tự chế đáng thử:

- **Min tiền tố / hậu tố** thay cho top-2: `min_except(prev, c) = min(min(prev[0..c-1]), min(prev[c+1..k-1]))`. Tính trước hai mảng `prefix`, `suffix` cho cả hàng trong `O(k)`. Cách này tổng quát hơn top-2 khi ràng buộc phức tạp hơn (ví dụ cấm cả một **đoạn** màu liên tiếp).
- **Dãy nhà xếp thành vòng tròn** (nhà `n - 1` kề nhà `0`): cố định màu `f` của nhà 0 (`k` lựa chọn), chạy DP `O(nk)` với hàng đầu chỉ cho phép màu `f`, rồi lấy min hàng cuối trên các màu `!= f`. Tổng `O(nk²)`.

In [30]:
def min_except_prefix_suffix(prev):
    """Trả về list r với r[c] = min(prev[x] với x != c), dùng min tiền tố / hậu tố, O(k)."""
    k = len(prev)
    prefix = [INF] * (k + 1)          # prefix[c] = min(prev[0..c-1])
    suffix = [INF] * (k + 1)          # suffix[c] = min(prev[c..k-1])
    for c in range(k):
        prefix[c + 1] = min(prefix[c], prev[c])
    for c in range(k - 1, -1, -1):
        suffix[c] = min(suffix[c + 1], prev[c])
    return [min(prefix[c], suffix[c + 1]) for c in range(k)]


for prev in [[8, 3, 6, 5], [4, 2, 2, 7], [7, 4, 9, 4]]:
    r = min_except_prefix_suffix(prev)
    assert r == [min_except_slow(prev, c) for c in range(len(prev))]
    print(f"prev = {prev} -> min_except = {r}")

random.seed(3)
for _ in range(1000):
    prev = [random.randint(1, 5) for _ in range(random.randint(2, 7))]
    assert min_except_prefix_suffix(prev) == [min_except_slow(prev, c) for c in range(len(prev))]
print("Min tiền tố / hậu tố khớp cách chậm trên 1000 hàng ngẫu nhiên")

prev = [8, 3, 6, 5] -> min_except = [3, 5, 3, 3]
prev = [4, 2, 2, 7] -> min_except = [2, 2, 2, 2]
prev = [7, 4, 9, 4] -> min_except = [4, 4, 4, 4]
Min tiền tố / hậu tố khớp cách chậm trên 1000 hàng ngẫu nhiên


In [31]:
def min_cost_circular(costs):
    """Biến thể: nhà n-1 kề nhà 0. Cố định màu nhà 0 rồi chạy DP top-2. O(n k^2)."""
    n, k = len(costs), len(costs[0])
    if n == 1:
        return min(costs[0])
    best = INF
    for first in range(k):
        prev = [costs[0][c] if c == first else INF for c in range(k)]   # nhà 0 chỉ được màu first
        for i in range(1, n):
            min1, min2, idx1 = two_smallest(prev)
            prev = [costs[i][c] + (min2 if c == idx1 else min1) for c in range(k)]
        best = min(best, min(prev[c] for c in range(k) if c != first))
    return best


def circular_bruteforce(costs):
    n, k = len(costs), len(costs[0])
    best = INF
    for colors in itertools.product(range(k), repeat=n):
        ok = all(a != b for a, b in zip(colors, colors[1:])) and (n == 1 or colors[0] != colors[-1])
        if ok:
            best = min(best, sum(costs[i][colors[i]] for i in range(n)))
    return best


cs = [[1, 9, 9], [9, 1, 9], [1, 9, 9]]       # dãy thẳng: 1+1+1 = 3; vòng tròn: nhà 0 và nhà 2 không được cùng màu 0
print("Dãy thẳng :", min_cost_o1(cs), "| vòng tròn:", min_cost_circular(cs))
random.seed(11)
for _ in range(300):
    n, k = random.randint(1, 6), random.randint(2, 4)
    cs = [[random.randint(1, 9) for _ in range(k)] for _ in range(n)]
    assert min_cost_circular(cs) == circular_bruteforce(cs), cs
print("Biến thể vòng tròn khớp brute force trên 300 input (kể cả trường hợp không có cách tô -> inf)")

Dãy thẳng : 3 | vòng tròn: 11
Biến thể vòng tròn khớp brute force trên 300 input (kể cả trường hợp không có cách tô -> inf)


## 20. Bài tập vận dụng

Làm theo thứ tự, **tự làm trước** rồi mới xem gợi ý (mục 21) và đáp án (mục 22). Các cell TODO bên dưới chỉ **định nghĩa** hàm; test được để sẵn dưới dạng comment — làm xong thì bỏ dấu `#` để tự kiểm tra.

### Bài 1 — Tính tay bảng DP với `min1, min2, idx1`

Cho `costs_hw = [[4, 9, 2, 7], [3, 1, 8, 6], [5, 4, 3, 9]]` (3 nhà, 4 màu). Không chạy code, hãy lập bảng như mục 11:

| nhà `i` | `costs[i]` | bộ ba hàng trên | `dp[i]` | bộ ba mới `(min1, min2, idx1)` |
|---:|---|---|---|---|
| 0 | `[4, 9, 2, 7]` | — | ? | ? |
| 1 | `[3, 1, 8, 6]` | ? | ? | ? |
| 2 | `[5, 4, 3, 9]` | ? | ? | ? |

Cho biết đáp án, và lần ngược ra một cách tô đạt đáp án đó.

### Bài 2 — Đoán output (không chạy code)

Dùng các hàm đã định nghĩa trong notebook, đoán kết quả của từng dòng:

```python
print(two_smallest([6, 2, 9, 2]))                          # (a)
print(two_smallest([5, 5]))                                # (b)
print(two_smallest([9, 4, 7, 1, 8]))                       # (c)
print([min_except_fast([4, 6, 1, 3], c) for c in range(4)])  # (d)
print(min_cost_two_min([[1, 100], [1, 100], [1, 100]]))    # (e)
```

### Bài 3 — Điền TODO: tìm 2 giá trị nhỏ nhất trong một lần duyệt

Viết lại `two_smallest` **không dùng** `sort`, `sorted`, `min`: chỉ một vòng `for` và các phép so sánh. Trả về `(min1, min2, idx1)` với `min2` là "nhỏ nhì tính cả lặp lại".

In [32]:
def exercise_two_smallest(values):
    # TODO 1: khởi tạo min1, min2 bằng vô cùng, idx1 = -1
    # TODO 2: duyệt j, x bằng enumerate(values)
    # TODO 3: nếu x < min1: vàng cũ xuống bạc, x lên vàng (nhớ cập nhật idx1)
    # TODO 4: ngược lại nếu x < min2: x lên bạc
    # TODO 5: trả về (min1, min2, idx1)
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_two_smallest([8, 3, 6, 5]) == (3, 5, 1)
# assert exercise_two_smallest([4, 2, 2, 7]) == (2, 2, 1)
# assert exercise_two_smallest([5, 2]) == (2, 5, 1)
# assert exercise_two_smallest([5, 5]) == (5, 5, 0)

### Bài 4 — Tự viết lại lời giải `O(nk)` thời gian, `O(1)` bộ nhớ thêm

Không nhìn lại mục 13, tự viết `exercise_min_cost_ii(costs)`. Yêu cầu: không tạo list độ dài `k` hay `n` nào, không sửa `costs`.

In [33]:
def exercise_min_cost_ii(costs: List[List[int]]) -> int:
    # TODO 1: bộ ba của "nhà ảo" trước nhà 0
    # TODO 2: với mỗi hàng: khởi tạo bộ ba đang xây = (INF, INF, -1)
    # TODO 3: với mỗi màu c: total = cost + (min2 hoặc min1 của nhà trước)
    # TODO 4: cập nhật bộ ba đang xây bằng total (vàng / bạc)
    # TODO 5: hết vòng màu mới chuyển giao bộ ba; cuối cùng trả về min1
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_min_cost_ii([[1, 5, 3], [2, 9, 4]]) == 5
# assert exercise_min_cost_ii([[1, 3], [2, 4]]) == 5
# assert exercise_min_cost_ii(costs_demo) == 5
# assert exercise_min_cost_ii([[1, 2, 30], [1, 30, 30]]) == 3
# random.seed(0)
# for _ in range(200):
#     n, k = random.randint(1, 5), random.randint(2, 4)
#     cs = [[random.randint(1, 5) for _ in range(k)] for _ in range(n)]
#     assert exercise_min_cost_ii(cs) == min_cost_bruteforce(cs)

### Bài 5 — Biến thể: trả về cả cách tô

Viết `exercise_min_cost_with_colors(costs)` trả về cặp `(chi_phí_nhỏ_nhất, colors)` trong đó `colors` là list màu của từng nhà (một cách tô đạt chi phí nhỏ nhất; nếu có nhiều cách, trả cách nào cũng được). Thời gian `O(nk)`.

Gợi ý nhỏ: lần này **cần giữ lại cả bảng** `dp` (bộ nhớ `O(nk)`), rồi **lần ngược** từ nhà cuối về nhà đầu như phần "Nhận xét về trace" ở mục 11.

In [34]:
def exercise_min_cost_with_colors(costs: List[List[int]]):
    # TODO 1: tính đủ bảng dp (n hàng), mỗi hàng dùng quy tắc min1 / min2 / idx1
    # TODO 2: màu nhà cuối = màu có dp nhỏ nhất ở hàng cuối
    # TODO 3: đi i = n-2 về 0: màu nhà i = màu c != màu nhà i+1 có dp[i][c] nhỏ nhất
    # TODO 4: trả về (dp[n-1][màu nhà cuối], colors)
    pass

# Tự làm xong rồi mới mở test:
# cost, colors = exercise_min_cost_with_colors(costs_demo)
# assert cost == 5 and len(colors) == 4
# assert all(a != b for a, b in zip(colors, colors[1:]))                  # hợp lệ
# assert sum(costs_demo[i][colors[i]] for i in range(4)) == cost          # tổng khớp

### Bài 6 — Mini challenge: LeetCode 1289 — Minimum Falling Path Sum II

Đề (tóm tắt): cho lưới `grid` kích thước `n x n`, hãy trả về tổng nhỏ nhất của một **"falling path with non-zero shifts"**: chọn **đúng một phần tử ở mỗi hàng** sao cho hai phần tử được chọn ở **hai hàng kề nhau không nằm cùng cột**. Constraints: `n == grid.length == grid[i].length`, `1 <= n <= 200`, `-99 <= grid[i][j] <= 99`.

Signature LeetCode: `def minFallingPathSum(self, grid: List[List[int]]) -> int`.

- Example 1: `grid = [[1,2,3],[4,5,6],[7,8,9]]` → `13` (chọn `1, 5, 7`).
- Example 2: `grid = [[7]]` → `7`.

Nhận ra đây là bài nào trong trang phục mới, rồi giải trong `O(n²)`. Cẩn thận: giá trị có thể **âm**.

In [35]:
def exercise_min_falling_path_sum_ii(grid: List[List[int]]) -> int:
    # TODO: hàng = ?, cột = ?, "không cùng cột ở hai hàng kề" = ?
    # TODO: viết lời giải O(n^2) (n hàng x n cột)
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_min_falling_path_sum_ii([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == 13
# assert exercise_min_falling_path_sum_ii([[7]]) == 7
# assert exercise_min_falling_path_sum_ii([[-1, -2], [-3, -4]]) == -5

## 21. Gợi ý theo mức

Mở từng mức một — chỉ đọc mức tiếp theo khi thực sự bí.

**Bài 1**
- Mức 1: hàng 0 thì `dp[0] = costs[0]`; bộ ba của nó tìm bằng mắt: số nhỏ nhất, vị trí của nó, số nhỏ nhì.
- Mức 2: ở hàng 1, chỉ **một** màu (màu `idx1` của hàng 0) phải cộng `min2`; ba màu còn lại cộng `min1`.
- Mức 3: hàng 0 có bộ ba `(2, 4, 2)`. Hàng 1: màu 2 cộng 4, các màu khác cộng 2.

**Bài 2**
- Mức 1: nhớ so sánh **nghiêm ngặt** `<` — số bằng `min1` đến sau **không** lên vàng mà đi xét ghế bạc.
- Mức 2: (d) bộ ba của `[4, 6, 1, 3]` là gì? Chỉ vị trí `idx1` nhận `min2`.
- Mức 3: (e) với `k = 2`, màu bắt buộc xen kẽ: chỉ có 2 cách tô `0,1,0` và `1,0,1`.

**Bài 3**
- Mức 1: hình dung huy chương vàng / bạc ở "Kiến thức nền 6".
- Mức 2: thứ tự quan trọng: `min2 = min1` phải chạy **trước** `min1 = x`. Hoặc gán đồng thời `min1, min2, idx1 = x, min1, j`.
- Mức 3: `if x < min1: ... elif x < min2: ...` — `elif`, không phải `if`.

**Bài 4**
- Mức 1: nhà ảo `(0, 0, -1)`; bộ ba đang xây bắt đầu `(INF, INF, -1)`.
- Mức 2: trong vòng màu: `total = cost + (prev_min2 if c == prev_idx1 else prev_min1)`, rồi đưa `total` vào cuộc thi vàng / bạc.
- Mức 3: dòng `prev_min1, prev_min2, prev_idx1 = cur_min1, cur_min2, cur_idx1` đặt **ngoài** vòng `for c`, **trong** vòng `for row`.

**Bài 5**
- Mức 1: `dp[i+1][x] = costs[i+1][x] + min(dp[i][c] với c != x)`. Biết màu nhà `i + 1` là `x`, màu nhà `i` là `c != x` làm min đó.
- Mức 2: lưu `dp` là list các hàng: `dp.append(cur)`.
- Mức 3: `colors[i] = min((c for c in range(k) if c != colors[i + 1]), key=lambda c: dp[i][c])`.

**Bài 6**
- Mức 1: hàng của lưới = nhà, cột = màu, `grid[i][j]` = chi phí. "Không cùng cột ở hai hàng kề" = "hai nhà kề không cùng màu".
- Mức 2: code gần như y hệt `minCostII`. Với `n = 1` thì `k = 1` — vẫn chạy đúng vì không có hàng thứ hai cần `min2`.
- Mức 3: giá trị âm không sao, miễn là bộ ba đang xây khởi tạo bằng `INF` (không phải 0). Nhà ảo `(0, 0, -1)` vẫn đúng vì nó là "chi phí trước khi chọn gì" = 0.

## 22. Đáp án tham khảo

Hãy tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

| nhà `i` | `costs[i]` | bộ ba hàng trên | `dp[i]` | bộ ba mới `(min1, min2, idx1)` |
|---:|---|---|---|---|
| 0 | `[4, 9, 2, 7]` | — | `[4, 9, 2, 7]` | `(2, 4, 2)` |
| 1 | `[3, 1, 8, 6]` | `(2, 4, 2)` | `[3+2, 1+2, 8+4, 6+2] = [5, 3, 12, 8]` | `(3, 5, 1)` |
| 2 | `[5, 4, 3, 9]` | `(3, 5, 1)` | `[5+3, 4+5, 3+3, 9+3] = [8, 9, 6, 12]` | `(6, 8, 2)` |

Đáp án = **6**. Lần ngược: nhà 2 màu 2 (`dp = 6`) → nhà 1 khác màu 2, rẻ nhất là màu 1 (`dp = 3`) → nhà 0 khác màu 1, rẻ nhất là màu 2 (`dp = 2`). Cách tô `[2, 1, 2]`: `2 + 1 + 3 = 6` ✓ (nhà 0 và nhà 2 cùng màu 2 là **hợp lệ** vì không kề nhau).

### Đáp án Bài 2

- (a) `(2, 2, 1)` — số 2 ở vị trí 3 không lên vàng (vì `2 < 2` sai) nhưng chiếm ghế bạc.
- (b) `(5, 5, 0)` — tương tự, hoà.
- (c) `(1, 4, 3)` — vàng đổi chủ 2 lần: 9 → 4 → 1; bạc cuối cùng là 4.
- (d) `[1, 1, 3, 1]` — bộ ba `(1, 3, 2)`: chỉ vị trí 2 nhận `min2 = 3`.
- (e) `102` — cách tô `0, 1, 0`: `1 + 100 + 1`.

In [36]:
# Kiểm chứng Bài 1 và Bài 2 bằng code
costs_hw = [[4, 9, 2, 7], [3, 1, 8, 6], [5, 4, 3, 9]]
assert trace_two_min(costs_hw) == 6

assert two_smallest([6, 2, 9, 2]) == (2, 2, 1)
assert two_smallest([5, 5]) == (5, 5, 0)
assert two_smallest([9, 4, 7, 1, 8]) == (1, 4, 3)
assert [min_except_fast([4, 6, 1, 3], c) for c in range(4)] == [1, 1, 3, 1]
assert min_cost_two_min([[1, 100], [1, 100], [1, 100]]) == 102
print("Đáp án Bài 1 và Bài 2: khớp!")

nhà 0: costs=[4, 9, 2, 7] -> dp=[4, 9, 2, 7] | (min1, min2, idx1) = (2, 4, 2)
nhà 1: costs=[3, 1, 8, 6], hàng trên có min1=2, min2=4, idx1=2
   c=0: 3 + min1(2) = 5
   c=1: 1 + min1(2) = 3
   c=2: 8 + min2(4) = 12   <- c trùng idx1, phải dùng min2
   c=3: 6 + min1(2) = 8
   -> dp=[5, 3, 12, 8] | (min1, min2, idx1) = (3, 5, 1)
nhà 2: costs=[5, 4, 3, 9], hàng trên có min1=3, min2=5, idx1=1
   c=0: 5 + min1(3) = 8
   c=1: 4 + min2(5) = 9   <- c trùng idx1, phải dùng min2
   c=2: 3 + min1(3) = 6
   c=3: 9 + min1(3) = 12
   -> dp=[8, 9, 6, 12] | (min1, min2, idx1) = (6, 8, 2)
Đáp án = min1 của hàng cuối = 6
Đáp án Bài 1 và Bài 2: khớp!


In [37]:
def answer_two_smallest(values):
    min1, min2, idx1 = float("inf"), float("inf"), -1
    for j, x in enumerate(values):
        if x < min1:
            min1, min2, idx1 = x, min1, j        # gán đồng thời: min2 nhận min1 CŨ
        elif x < min2:
            min2 = x
    return min1, min2, idx1


def answer_min_cost_ii(costs: List[List[int]]) -> int:
    prev_min1, prev_min2, prev_idx1 = 0, 0, -1
    for row in costs:
        cur_min1, cur_min2, cur_idx1 = float("inf"), float("inf"), -1
        for c, cost in enumerate(row):
            total = cost + (prev_min2 if c == prev_idx1 else prev_min1)
            if total < cur_min1:
                cur_min1, cur_min2, cur_idx1 = total, cur_min1, c
            elif total < cur_min2:
                cur_min2 = total
        prev_min1, prev_min2, prev_idx1 = cur_min1, cur_min2, cur_idx1
    return prev_min1


def answer_min_cost_with_colors(costs: List[List[int]]):
    n, k = len(costs), len(costs[0])
    dp = [costs[0][:]]
    for i in range(1, n):
        min1, min2, idx1 = answer_two_smallest(dp[-1])
        dp.append([costs[i][c] + (min2 if c == idx1 else min1) for c in range(k)])
    colors = [0] * n
    colors[-1] = min(range(k), key=lambda c: dp[-1][c])
    for i in range(n - 2, -1, -1):           # lần ngược: nhà i khác màu nhà i+1, dp nhỏ nhất
        colors[i] = min((c for c in range(k) if c != colors[i + 1]), key=lambda c: dp[i][c])
    return dp[-1][colors[-1]], colors


def answer_min_falling_path_sum_ii(grid: List[List[int]]) -> int:
    # Hàng = nhà, cột = màu: chính là minCostII (giá trị âm không ảnh hưởng)
    return answer_min_cost_ii(grid)


# Test Bài 3
assert answer_two_smallest([8, 3, 6, 5]) == (3, 5, 1)
assert answer_two_smallest([4, 2, 2, 7]) == (2, 2, 1)
assert answer_two_smallest([5, 2]) == (2, 5, 1)
assert answer_two_smallest([5, 5]) == (5, 5, 0)
# Test Bài 4
assert answer_min_cost_ii([[1, 5, 3], [2, 9, 4]]) == 5
assert answer_min_cost_ii([[1, 3], [2, 4]]) == 5
assert answer_min_cost_ii(costs_demo) == 5
assert answer_min_cost_ii([[1, 2, 30], [1, 30, 30]]) == 3
# Test Bài 5
cost, colors = answer_min_cost_with_colors(costs_demo)
print("Bài 5 - costs_demo:", cost, colors)
assert (cost, colors) == (5, [1, 2, 3, 1])
# Test Bài 6
assert answer_min_falling_path_sum_ii([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == 13
assert answer_min_falling_path_sum_ii([[7]]) == 7
assert answer_min_falling_path_sum_ii([[-1, -2], [-3, -4]]) == -5

# Đối chiếu ngẫu nhiên với brute force
random.seed(2026)
for _ in range(400):
    n, k = random.randint(1, 6), random.randint(2, 4)
    cs = [[random.randint(1, 6) for _ in range(k)] for _ in range(n)]
    expected = min_cost_bruteforce(cs)
    assert answer_min_cost_ii(cs) == expected, cs
    cost, colors = answer_min_cost_with_colors(cs)
    assert cost == expected and len(colors) == n, cs
    assert all(a != b for a, b in zip(colors, colors[1:])), cs
    assert sum(cs[i][colors[i]] for i in range(n)) == cost, cs
    g = [[random.randint(-99, 99) for _ in range(k)] for _ in range(k)]   # lưới vuông k x k cho Bài 6
    assert answer_min_falling_path_sum_ii(g) == min_cost_bruteforce(g), g
    vals = [random.randint(1, 4) for _ in range(k)]
    assert answer_two_smallest(vals) == two_smallest(vals)
print("Đáp án tham khảo Bài 3–6: PASS toàn bộ test!")

Bài 5 - costs_demo: 5 [1, 2, 3, 1]


Đáp án tham khảo Bài 3–6: PASS toàn bộ test!


## 23. Template ngắn nên nhớ

```python
class Solution:
    def minCostII(self, costs: List[List[int]]) -> int:
        min1, min2, idx1 = 0, 0, -1                   # "nhà ảo": chi phí 0, không màu
        for row in costs:
            n1, n2, ni = float("inf"), float("inf"), -1
            for c, cost in enumerate(row):
                total = cost + (min2 if c == idx1 else min1)   # trùng màu rẻ nhất -> lấy rẻ nhì
                if total < n1:
                    n1, n2, ni = total, n1, c            # vàng cũ xuống bạc
                elif total < n2:
                    n2 = total
            min1, min2, idx1 = n1, n2, ni                 # chuyển giao SAU vòng màu
        return min1
```

Câu thần chú: **"Muốn tô màu `c`: nếu `c` không phải màu của phương án rẻ nhất nhà trước thì cộng `min1`; nếu trùng thì cộng `min2`."**

Công thức gốc cần nhớ để tự dựng lại mọi thứ: `dp[i][c] = costs[i][c] + min(dp[i-1][c'] với c' != c)`.

## 24. Checklist trước khi submit

- Đúng tên method `minCostII` và signature `(self, costs: List[List[int]]) -> int`?
- Lấy min hàng trên **có loại màu `c`** (dùng `min2` khi `c == idx1`)?
- Tóm tắt **hàng dp** (chi phí tích luỹ), không phải `costs` thô của nhà trước?
- Cập nhật top-2: `min2 = min1` **trước** `min1 = x`; dùng **`elif`**; so sánh nghiêm ngặt `<`?
- Bộ ba đang xây khởi tạo bằng `inf`; chỉ chuyển giao **sau** vòng duyệt màu?
- Trả về `min1` của hàng cuối (tức `min(dp[-1])`), không phải một ô cố định?
- Đã thử: 2 ví dụ đề (`5`, `5`), `n = 1`, `k = 2`, hàng có hoà (`costs_demo` → `5`), bẫy tham lam (`[[1,2,30],[1,30,30]]` → `3`)?
- Không sửa input `costs` (trừ khi chắc chắn được phép)?